# Pipeline OCR — Partie 1 V13.7.1 Evidence-First + Rotation Safe

Extraction réglementaire. Correction ciblée TTR_DUREE, conservation des dates TTR valides, chemins V12 figés.


## FINAL V12 — choix d'architecture

Cette version conserve le contrat RAW `DOM_EXTRACTION_V1` et les **99 champs** de la V9.2, mais renforce la lecture des formulaires pré-imprimés.

Principes FINAL V12 :
- aucune normalisation métier dans la Partie 1 ;
- classification VLM obligatoire par page ;
- prise en compte d'un **décalage vertical global** des valeurs par rapport aux libellés, vers le haut ou vers le bas ;
- si un champ critique manque, si une valeur est sémantiquement incompatible avec son champ (ex. `TTR_DATE_DEBUT = ORAN`) ou si des champs liés sont incohérents, **relecture de toute la page concernée**, jamais du document entier ;
- recovery 1 en HD 2000 px, puis recovery 2 en 2400 px si le problème persiste ou si une correction doit être confirmée ;
- aucune permutation automatique de valeurs : en cas d'ambiguïté persistante, la valeur reste signalée pour revue ;
- ajout d'un **indicateur de confiance opérationnel** par champ et par page, fondé sur validité sémantique + accord entre lectures + résolution des retries. Il ne s'agit pas d'une probabilité native Qwen/log-prob.


## 1. Dépendances


In [ ]:
# Si nécessaire sur un environnement neuf Domino :
# %pip install -q -U 'transformers>=4.57.0' accelerate pymupdf pillow pandas psutil
#
# IMPORTANT : aucune dépendance flash-attn n'est requise ni utilisée.


## 2. Imports


In [ ]:
import gc
import hashlib
import json
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import fitz
import numpy as np
import pandas as pd
import psutil
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

print('✅ Imports OK')
print('Python      :', sys.version.split()[0])
print('Torch       :', torch.__version__)
print('CUDA dispo :', torch.cuda.is_available())
print('GPU        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun')


## 3. Configuration V9


In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---------- Contrat de données ----------
SCHEMA_VERSION = 'DOM_EXTRACTION_V1'
PIPELINE_VERSION = 'GENERIC_V13_7_1_PART1_SINGLE_RECOVERY_PAGE_TIMING_FIXED'
FIELD_SCHEMA_HASH = 'da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e'

# ---------- Test ----------
MAX_PDFS = None     # production: tous les PDF; mettre 10 uniquement pour un test
RESUME = True       # un JSON V9.2 n'est PAS repris : pipeline_version différent

# ---------- Images ----------
PDF_ZOOM = 2.0
IMAGE_MAX_SIZE = 1400
IMAGE_MAX_SIZE_CLASSIFICATION = 1100
PDF_ZOOM_HAUTE_DEF = 4
IMAGE_MAX_SIZE_HAUTE_DEF = 1800
IMAGE_MAX_SIZE_PERMIS_RETRY = 2000
IMAGE_MAX_SIZE_RECOVERY_1 = 2000
IMAGE_MAX_SIZE_RECOVERY_2 = 2400
MIN_PIXELS = 4 * 32 * 32
MAX_PIXELS = 2400 * 32 * 32
BLANK_THRESHOLD = 0.95
CLASSIFICATION_THRESHOLD = 0.90
CLASSIFICATION_RETRY_ON_AUTRE = True
CLASSIFICATION_RETRY_LOW_CONFIDENCE = True
CLASSIFICATION_HARD_MIN_CONFIDENCE = 0.90
BLOCK_EXTRACTION_ON_CLASSIFICATION_CONFLICT = True

# ---------- Batch ----------
GPU_BATCH_SIZE_CLASSIFICATION = 16
GPU_BATCH_SIZE_EXTRACTION_STANDARD = 4
GPU_BATCH_SIZE_EXTRACTION_HD = 2
MAX_NEW_TOKENS_CLASSIFICATION = 100
MAX_NEW_TOKENS_EXTRACTION = 1700
MAX_NEW_TOKENS_RECOVERY = 1900

# ---------- V13.5 : budget génération par type ----------
# Les sorties s'arrêtent normalement sur EOS avant ces plafonds.
# Les plafonds servent surtout de garde-fou contre une génération anormalement longue.
MAX_NEW_TOKENS_BY_DOC = {
    'ENGAGEMENT_DOMICILIATION': 900,
    'CONTRAT_TRAVAIL': 1300,
    'CONTRAT_SPECIFIQUE': 1500,
    'TITRE_TRAVAIL': 1600,
    'PERMIS_TRAVAIL_COUVERTURE': 400,
}
MAX_NEW_TOKENS_RECOVERY_BY_DOC = {
    'ENGAGEMENT_DOMICILIATION': 1100,
    'CONTRAT_TRAVAIL': 1400,
    'CONTRAT_SPECIFIQUE': 1600,
    'TITRE_TRAVAIL': 1700,
    'PERMIS_TRAVAIL_COUVERTURE': 500,
}
MAX_NEW_TOKENS_TTR_STRUCTURAL = 450

# Sortie Qwen compacte : les 99 champs finaux restent CANONIQUES dans raw_data.
COMPACT_OUTPUT_ENABLED = True

# ---------- Extraction / recovery page entière ----------
SEUIL_REMPLISSAGE_OK = 0.55
SEUIL_REMPLISSAGE_MIN = 0.40
ENABLE_PAGE_RECOVERY = True
ENABLE_RECOVERY_CONFIRMATION = False

# V13.3 Fast-Safe : aucun compromis sur modèle/résolution.
FAST_SAFE_TTR_INITIAL_LOCK = True
FAST_SAFE_CONFIRM_CRITICAL_ONLY = True
ENABLE_VIRTUAL_PERMIT_COVER = False
VIRTUAL_COVER_MIN_INK_RATIO = 0.025

# V13.7 : une seule relecture de sécurité.
# INITIAL -> contrôles Python -> si nécessaire -> PAGE ENTIERE 2400 px -> FINAL.
# Aucun recovery 2000 et aucune confirmation 2400 supplémentaire.
RECOVERY_ON_CRITICAL_MISSING = True
RECOVERY_ON_SEMANTIC_MISMATCH = False
RECOVERY_ON_COHERENCE_ERROR = False  # cohérence métier réservée à la Partie 2
RECOVERY_ON_LOW_FILL = True

# ---------- Confidence opérationnelle ----------
# Ce score est un indicateur explicable de robustesse de l'extraction.
# Il n'est PAS une probabilité native/log-prob Qwen.
CONFIDENCE_METHOD = 'OPERATIONAL_CONSENSUS_SEMANTIC_V1'
CONFIDENCE_HIGH = 90
CONFIDENCE_MEDIUM = 75

# ---------- Audit / transparence des appels Qwen ----------
STORE_QWEN_RAW_TEXT_IN_ATTEMPTS = True
STORE_PARSED_DATA_IN_ATTEMPTS = False
STORE_AGGREGATED_EXTRACTION_RAW_TEXT = False
PRINT_CALL_DIAGNOSTICS = True

# ---------- Fichiers ----------
INPUT_DIR = Path('/mnt/data/domiciliations_in')
OUTPUT_ROOT = Path('/mnt/data/document_pipeline/V13_7_1')
RAW_ROOT = OUTPUT_ROOT / '01_extraction_raw'
JSON_DIR = RAW_ROOT / 'json_dossiers'
LOG_PATH = RAW_ROOT / 'pipeline_extraction_v13_7.log'
MANIFEST_PATH = RAW_ROOT / 'extraction_manifest_v13_7_1.json'
INDEX_CSV_PATH = RAW_ROOT / 'extraction_index_v13_7_1.csv'

INPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.glob('*.pdf'))
if MAX_PDFS is not None:
    pdfs = pdfs[:int(MAX_PDFS)]

print('Pipeline       :', PIPELINE_VERSION)
print('Schema         :', SCHEMA_VERSION)
print('Schema hash    :', FIELD_SCHEMA_HASH[:16] + '…')
print('PDFs sélectionnés :', len(pdfs))
print('Entrée         :', INPUT_DIR)
print('Sortie RAW     :', JSON_DIR)
print('Recovery page  : UNIQUE', IMAGE_MAX_SIZE_RECOVERY_2, 'px')
print('Couverture PTR : extraction Qwen DESACTIVEE')
print('Confidence     :', CONFIDENCE_METHOD)


## 4. Chargement Qwen — sans Flash-Attention


In [ ]:
if DEVICE != 'cuda':
    raise RuntimeError('Ce pipeline nécessite un GPU CUDA.')

torch.backends.cuda.matmul.allow_tf32 = True

print('Chargement du processor...')
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = 'left'

print('Chargement du modèle FP8...')
print('ℹ️ V9 : aucune dépendance flash-attn / flash_attention_2.')
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

# Chargement volontairement aligné sur la V7.2/V8.1 qui fonctionne sur Domino.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()

print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')
print(f'VRAM allouée : {torch.cuda.memory_allocated()/1e9:.2f} GB')
_device_map = getattr(model, 'hf_device_map', None)
if _device_map:
    print('Device map    :', _device_map)
    _offload = [v for v in _device_map.values() if str(v).lower() in {'cpu','disk'}]
    if _offload:
        print('⚠️ Offload CPU/disk détecté : inférence potentiellement ralentie.')


## 5. Utilitaires PDF / image / JSON


In [ ]:

# =====================================================================
# V13.2 — ORIENTATION / DESKEW AVANT VLM
# =====================================================================
# But :
# - corriger les petites inclinaisons sans consommer de tokens ;
# - conserver l'image originale intacte ;
# - ne jamais faire de rotation 90/180/270 sur une simple heuristique fragile.
#
# La petite inclinaison est estimée à partir des lignes de texte/structures.
# Une correction n'est appliquée que si l'angle est plausible et suffisamment net.

import cv2
import numpy as np
from PIL import Image

DESKEW_ENABLED = True
DESKEW_MIN_ABS_DEG = 0.35
DESKEW_MAX_ABS_DEG = 7.0
DESKEW_MIN_LINES = 6

def _pil_to_cv_gray(img):
    arr=np.asarray(img.convert("RGB"))
    return cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)

def estimate_small_skew_deg(img):
    """
    Retourne l'angle dominant des lignes quasi horizontales.
    Valeur >0 : contenu incliné dans le sens anti-horaire.
    None : preuve insuffisante.
    """
    gray=_pil_to_cv_gray(img)
    h,w=gray.shape[:2]
    scale=min(1.0, 1600.0/max(h,w))
    if scale < 1.0:
        gray=cv2.resize(gray, (int(w*scale),int(h*scale)), interpolation=cv2.INTER_AREA)

    # Texte sombre sur fond clair -> contours.
    blur=cv2.GaussianBlur(gray,(3,3),0)
    edges=cv2.Canny(blur,50,150,apertureSize=3)
    min_len=max(80, int(gray.shape[1]*0.12))
    lines=cv2.HoughLinesP(edges,1,np.pi/1800,threshold=60,
                          minLineLength=min_len,maxLineGap=20)
    if lines is None:
        return None, 0

    angles=[]
    for ln in lines[:,0,:]:
        x1,y1,x2,y2=map(float,ln)
        dx=x2-x1; dy=y2-y1
        if abs(dx)<1:
            continue
        a=np.degrees(np.arctan2(dy,dx))
        # garder uniquement les lignes proches de l'horizontale
        if -DESKEW_MAX_ABS_DEG <= a <= DESKEW_MAX_ABS_DEG:
            angles.append(a)

    if len(angles) < DESKEW_MIN_LINES:
        return None, len(angles)

    med=float(np.median(angles))
    mad=float(np.median(np.abs(np.asarray(angles)-med))) if angles else 999.0
    # dispersion excessive = pas de correction automatique
    if mad > 1.8:
        return None, len(angles)
    return med, len(angles)

def rotate_pil_expand_white(img, angle_deg):
    arr=np.asarray(img.convert("RGB"))
    h,w=arr.shape[:2]
    center=(w/2.0,h/2.0)
    M=cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    cos=abs(M[0,0]); sin=abs(M[0,1])
    nw=int(h*sin+w*cos); nh=int(h*cos+w*sin)
    M[0,2]+=nw/2-center[0]
    M[1,2]+=nh/2-center[1]
    rot=cv2.warpAffine(arr,M,(nw,nh),flags=cv2.INTER_CUBIC,
                       borderMode=cv2.BORDER_CONSTANT,borderValue=(255,255,255))
    return Image.fromarray(rot)

def deskew_for_vlm(img):
    meta={
        "deskew_enabled": bool(DESKEW_ENABLED),
        "deskew_applied": False,
        "deskew_detected_angle_deg": None,
        "deskew_correction_angle_deg": 0.0,
        "deskew_evidence_lines": 0,
    }
    if not DESKEW_ENABLED:
        return img,meta

    angle,nlines=estimate_small_skew_deg(img)
    meta["deskew_evidence_lines"]=int(nlines)
    if angle is None:
        return img,meta

    meta["deskew_detected_angle_deg"]=round(float(angle),3)
    if DESKEW_MIN_ABS_DEG <= abs(angle) <= DESKEW_MAX_ABS_DEG:
        # correction inverse
        corr=-float(angle)
        out=rotate_pil_expand_white(img,corr)
        meta["deskew_applied"]=True
        meta["deskew_correction_angle_deg"]=round(corr,3)
        return out,meta
    return img,meta


In [ ]:
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w*ratio), int(h*ratio)), Image.LANCZOS)


def white_ratio(image):
    arr = np.array(image.convert('L'))
    return float((arr > 245).sum() / arr.size)


def pdf_to_pages(path, zoom=PDF_ZOOM):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise ValueError(f'PDF absent ou vide : {path}')
    pages=[]
    doc=fitz.open(str(path))
    try:
        matrix=fitz.Matrix(zoom, zoom)
        for i in range(doc.page_count):
            page=doc.load_page(i)
            pix=page.get_pixmap(matrix=matrix, alpha=False)
            img=Image.frombytes('RGB',(pix.width,pix.height),pix.samples)
            img=resize_image(img)
            pages.append({
                'index':i, 'page_num':i+1, 'image':img,
                'width':img.width, 'height':img.height,
                'white_ratio':round(white_ratio(img),6),
            })
    finally:
        doc.close()
    return pages


def parse_json_response(text):
    if not text:
        return {}
    clean=str(text).strip()
    clean=re.sub(r'^```(?:json)?','',clean,flags=re.I).strip()
    clean=re.sub(r'```$','',clean).strip()
    match=re.search(r'\{.*\}',clean,flags=re.S)
    if not match:
        return {}
    candidate=match.group(0)
    for attempt in [candidate, re.sub(r',\s*([}\]])',r'\1',candidate)]:
        try:
            obj=json.loads(attempt)
            return obj if isinstance(obj,dict) else {}
        except Exception:
            pass
    return {}


def log(message):
    line=f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH,'a',encoding='utf-8') as f:
        f.write(line+'\n')


def crop_region(image, haut=0.0, bas=1.0, gauche=0.0, droite=1.0):
    largeur, hauteur=image.size
    x0=int(max(0,min(1,gauche))*largeur); x1=int(max(0,min(1,droite))*largeur)
    y0=int(max(0,min(1,haut))*hauteur); y1=int(max(0,min(1,bas))*hauteur)
    if x1<=x0 or y1<=y0:
        return image
    return image.crop((x0,y0,x1,y1))


def render_page_region(pdf_path, page_index, zoom=PDF_ZOOM_HAUTE_DEF,
                       max_side=IMAGE_MAX_SIZE_HAUTE_DEF, crop=None):
    doc=fitz.open(str(pdf_path))
    try:
        page=doc.load_page(int(page_index))
        pix=page.get_pixmap(matrix=fitz.Matrix(zoom,zoom),alpha=False)
        img=Image.frombytes('RGB',(pix.width,pix.height),pix.samples)
    finally:
        doc.close()
    if crop:
        img=crop_region(img,*crop)
    return resize_image(img,max_side=max_side)


FRONTIERE_ZONE_RECHERCHE=(0.28,0.66)
FRONTIERE_SEUIL_ENCRE=0.004
FRONTIERE_DEFAUT=0.47

def detecter_frontiere_documents(image, zone=FRONTIERE_ZONE_RECHERCHE,
                                  seuil_encre=FRONTIERE_SEUIL_ENCRE,
                                  defaut=FRONTIERE_DEFAUT):
    try:
        arr=np.array(image.convert('L'))
    except Exception:
        return defaut
    h=arr.shape[0]
    if h<10:
        return defaut
    densite=(arr<200).sum(axis=1)/max(arr.shape[1],1)
    y_min=int(max(0,zone[0])*h); y_max=int(min(1,zone[1])*h)
    bandes=[]; debut=None
    for y in range(y_min,y_max):
        vide=densite[y]<seuil_encre
        if vide and debut is None:
            debut=y
        elif not vide and debut is not None:
            bandes.append((debut,y)); debut=None
    if debut is not None:
        bandes.append((debut,y_max))
    if not bandes:
        return defaut
    debut,fin=max(bandes,key=lambda b:b[1]-b[0])
    if (fin-debut)/h<0.015:
        return defaut
    return round((debut+fin)/2/h,4)


def crops_planche_permis(image,marge=0.02):
    f=detecter_frontiere_documents(image)
    bas=min(1.0,f+marge); haut=max(0.0,f-marge)
    return {
        'frontiere':f,
        'titre':(0.00,bas,0.00,1.00),
        'colonne_identite':(0.00,bas,0.44,1.00),
        'colonne_poste':(0.00,bas,0.00,0.56),
        'couverture':(haut,1.00,0.00,1.00),
    }


def ink_ratio(image,threshold=200):
    arr=np.array(image.convert('L'))
    return float((arr<threshold).sum()/arr.size) if arr.size else 0.0


def image_for_classification(image):
    return resize_image(image,max_side=IMAGE_MAX_SIZE_CLASSIFICATION)


def is_missing_raw(value):
    # Sert uniquement à décider si Qwen doit relire un champ.
    # La valeur stockée dans raw_data n'est jamais normalisée par cette fonction.
    if value is None:
        return True
    if isinstance(value,str):
        t=value.strip()
        return (not t) or t.upper() in {'NULL','NONE','N/A','NA','ILLISIBLE','NON LISIBLE'}
    return False


def value_is_suspect(field,value):
    # V9 : contrôle d'extraction volontairement minimal.
    # Aucun parseur montant/date ici. Les formats sont traités en Partie 2.
    if is_missing_raw(value):
        return True
    text=str(value).strip()
    if field in {'TTR_NUMERO_PERMIS','CTR_NUMERO_PERMIS_TRAVAIL',
                 'CTS_NUMERO_PERMIS_TRAVAIL','PTR_NUMERO_SERIE'}:
        return len(re.sub(r'[^A-Za-z0-9]','',text)) < 5
    if field == 'DOM_COMPTE_LOCAL':
        return len(re.sub(r'\s+','',text)) < 8
    return False


def taux_remplissage(data,champs_attendus):
    if not champs_attendus:
        return 1.0
    n=sum(1 for c in champs_attendus if not is_missing_raw((data or {}).get(c)))
    return round(n/len(champs_attendus),4)

print('✅ Utilitaires V9 RAW OK')


## 6. Inférence GPU batch


In [ ]:

# =====================================================================
# V13.7 PROFILER — temps réel par étape ET par page
# =====================================================================
from contextlib import contextmanager

PROFILER_ENABLED = True
PROFILER_EVENTS = []
PAGE_PROFILER_ROWS = []
PROFILER_T0 = None

def _pnow():
    return time.perf_counter()

def _psync():
    try:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    except Exception:
        pass

def _pelapsed():
    return 0.0 if PROFILER_T0 is None else _pnow() - PROFILER_T0

def _pprint(msg):
    if PROFILER_ENABLED:
        print(f"[PROF {_pelapsed():8.3f}s] {msg}", flush=True)

def profiler_reset():
    global PROFILER_EVENTS, PAGE_PROFILER_ROWS, PROFILER_T0
    PROFILER_EVENTS = []
    PAGE_PROFILER_ROWS = []
    PROFILER_T0 = _pnow()
    _pprint("START DOSSIER")

def pevent(stage, seconds, **meta):
    PROFILER_EVENTS.append({
        "stage": stage,
        "elapsed_s": round(float(seconds), 6),
        **meta
    })

def page_pevent(page_num, doc_type, stage, strategy,
                shared_batch_s, batch_size=1,
                tokens_in=0, tokens_out=0, note=None):
    """Le temps exact d'un batch GPU est partagé entre ses pages.
    allocated_s = temps batch / nombre d'éléments, uniquement pour obtenir
    une imputation additive par page sans prétendre à un temps GPU isolé.
    """
    batch_size = max(1, int(batch_size or 1))
    row = {
        "page": page_num,
        "doc_type": doc_type,
        "stage": stage,
        "strategy": strategy,
        "batch_size": batch_size,
        "shared_batch_s": round(float(shared_batch_s), 3),
        "allocated_s": round(float(shared_batch_s) / batch_size, 3),
        "tokens_in": int(tokens_in or 0),
        "tokens_out": int(tokens_out or 0),
        "note": note or "",
    }
    PAGE_PROFILER_ROWS.append(row)
    _pprint(
        f"PAGE {page_num} | {doc_type} | {stage} | {strategy} | "
        f"batch={batch_size} temps_partage={row['shared_batch_s']:.3f}s "
        f"temps_impute={row['allocated_s']:.3f}s | "
        f"tokens={row['tokens_in']}+{row['tokens_out']}"
    )

@contextmanager
def pstage(stage, **meta):
    _psync()
    t = _pnow()
    _pprint(f"START {stage} | {meta}" if meta else f"START {stage}")
    try:
        yield
    finally:
        _psync()
        d = _pnow() - t
        pevent(stage, d, **meta)
        _pprint(f"END   {stage} | {d:.3f}s")

def profiler_finish(root):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)

    if PROFILER_EVENTS:
        df = pd.DataFrame(PROFILER_EVENTS)
        sm = (df.groupby("stage")
                .agg(appels=("stage", "size"), temps_s=("elapsed_s", "sum"))
                .reset_index()
                .sort_values("temps_s", ascending=False))
        total = float(sm.temps_s.sum()) or 1.0
        sm["pct_mesure"] = (sm.temps_s / total * 100).round(1)

        print("\n================ PROFILER GLOBAL =================")
        print(sm.to_string(index=False))
        print("==================================================")

        df.to_csv(root / "profiler_events_v13_7_1.csv",
                  index=False, encoding="utf-8-sig")
        sm.to_csv(root / "profiler_summary_v13_7_1.csv",
                  index=False, encoding="utf-8-sig")

    if PAGE_PROFILER_ROWS:
        pdf = pd.DataFrame(PAGE_PROFILER_ROWS)

        print("\n================ DETAIL PAR PAGE =================")
        cols = ["page","doc_type","stage","strategy","batch_size",
                "shared_batch_s","allocated_s","tokens_in","tokens_out","note"]
        print(pdf[cols].to_string(index=False))

        # Temps imputé = métrique additive. Le temps partagé du batch reste visible
        # dans le détail et ne doit pas être additionné page par page.
        ps = (pdf.groupby(["page","doc_type"], dropna=False)
                .agg(
                    qwen_etapes=("stage","size"),
                    temps_impute_s=("allocated_s","sum"),
                    tokens_in=("tokens_in","sum"),
                    tokens_out=("tokens_out","sum"),
                )
                .reset_index()
                .sort_values(["page","doc_type"]))

        print("\n================ RESUME PAR PAGE =================")
        print(ps.to_string(index=False))
        print("NOTE : 'temps_impute_s' répartit le temps d'un batch entre ses pages.")
        print("       'shared_batch_s' dans le détail est le vrai temps GPU du batch.")
        print("==================================================")

        pdf.to_csv(root / "profiler_pages_detail_v13_7_1.csv",
                   index=False, encoding="utf-8-sig")
        ps.to_csv(root / "profiler_pages_summary_v13_7_1.csv",
                  index=False, encoding="utf-8-sig")

        print("📊", root / "profiler_pages_detail_v13_7_1.csv")
        print("📊", root / "profiler_pages_summary_v13_7_1.csv")


In [ ]:
def apply_template(messages):
    """Qwen3 : désactive le mode 'thinking' si supporté."""
    try:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def ask_single(prompt, image, max_new_tokens):
    messages=[{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    with pstage("CHAT_TEMPLATE",batch=1):
        text_in=apply_template(messages)
    with pstage("PROCESSOR",batch=1):
        inputs=processor(text=[text_in],images=[image],return_tensors='pt')
    with pstage("TRANSFER_GPU",batch=1):
        inputs=inputs.to(DEVICE)
    tin=int(inputs['input_ids'].shape[1])
    _psync(); t=_pnow(); _pprint(f"START MODEL_GENERATE | batch=1 in={tin} max_new={max_new_tokens}")
    with torch.inference_mode():
        out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,
            repetition_penalty=1.0,use_cache=True,pad_token_id=processor.tokenizer.eos_token_id)
    _psync(); gen_s=_pnow()-t
    generated=out[0][inputs['input_ids'].shape[1]:]; tout=int(len(generated))
    pevent("MODEL_GENERATE",gen_s,batch=1,tokens_in=tin,tokens_out=tout,max_new_tokens=max_new_tokens)
    _pprint(f"END   MODEL_GENERATE | {gen_s:.3f}s | in={tin} out={tout}")
    with pstage("DECODE",batch=1,tokens_out=tout):
        text=processor.decode(generated,skip_special_tokens=True,clean_up_tokenization_spaces=True)
    return {'text':text,'tokens_in':tin,'tokens_out':tout,'elapsed_s':round(gen_s,3)}


def ask_batch_mixed(prompts, images, max_new_tokens):
    if not images: return []
    if len(prompts)!=len(images): raise ValueError('prompts et images doivent avoir la même longueur')
    if len(images)==1: return [ask_single(prompts[0],images[0],max_new_tokens)]
    texts_in=[]
    with pstage("CHAT_TEMPLATE",batch=len(images)):
        for prompt,image in zip(prompts,images):
            messages=[{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
            texts_in.append(apply_template(messages))
    with pstage("PROCESSOR",batch=len(images)):
        inputs=processor(text=texts_in,images=images,return_tensors='pt',padding=True)
    with pstage("TRANSFER_GPU",batch=len(images)):
        inputs=inputs.to(DEVICE)
    input_width=inputs['input_ids'].shape[1]
    attention_mask=inputs.get('attention_mask')
    total_in=int(attention_mask.sum().item()) if attention_mask is not None else int(input_width*len(images))
    _psync(); t=_pnow(); _pprint(f"START MODEL_GENERATE | batch={len(images)} in={total_in} max_new={max_new_tokens}")
    with torch.inference_mode():
        out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,
            repetition_penalty=1.0,use_cache=True,pad_token_id=processor.tokenizer.eos_token_id)
    _psync(); gen_s=_pnow()-t
    if out.shape[0]!=len(images): raise RuntimeError(f'Réponses VLM incohérentes : {out.shape[0]} sortie(s) pour {len(images)} image(s)')
    results=[]; total_out=0
    with pstage("DECODE",batch=len(images)):
        for i in range(len(images)):
            generated=out[i][input_width:]
            text=processor.decode(generated,skip_special_tokens=True,clean_up_tokenization_spaces=True)
            tin=int(attention_mask[i].sum().item()) if attention_mask is not None else int(input_width)
            tout=int(len(generated)); total_out+=tout
            results.append({'text':text,'tokens_in':tin,'tokens_out':tout,'elapsed_s':round(gen_s/len(images),3)})
    seq_out=[int(x.get('tokens_out',0) or 0) for x in results]
    pevent(
        "MODEL_GENERATE",gen_s,batch=len(images),tokens_in=total_in,
        tokens_out=total_out,sequence_tokens_out=str(seq_out),
        max_new_tokens=max_new_tokens
    )
    _pprint(
        f"END   MODEL_GENERATE | {gen_s:.3f}s | batch={len(images)} "
        f"in={total_in} out={total_out} | seq_out={seq_out}"
    )
    return results


def ask_batch(prompt, images, max_new_tokens):
    """Compatibilité avec la classification V7 : même prompt pour le batch."""
    return ask_batch_mixed([prompt] * len(images), images, max_new_tokens)


def is_cuda_oom(exc):
    text = str(exc).lower()
    return isinstance(exc, torch.cuda.OutOfMemoryError) or 'out of memory' in text


print('✅ Inférence V9 single / batch multi-prompts OK')


## 7. Prompts — mêmes 99 champs que V8.1


In [ ]:
PROMPT_CLASSIFICATION = """
Analyse le titre, les en-têtes, la mise en page et les blocs visuels
de cette page.

Classe la page dans exactement une seule catégorie :

- ENGAGEMENT_DOMICILIATION
- CONTRAT_TRAVAIL
- CONTRAT_SPECIFIQUE
- TITRE_TRAVAIL
- PERMIS_TRAVAIL_COUVERTURE
- AUTRE

RÈGLE DE PRIORITÉ ABSOLUE :
Certaines pages contiennent DEUX documents superposés : un titre de
travail bilingue dans la moitié haute et une couverture de permis de
travail dans la moitié basse.
Dans ce cas, classer TOUJOURS la page en TITRE_TRAVAIL.
Le grand titre « جواز العمل / Permis de Travail » de la moitié basse ne
doit JAMAIS l'emporter sur un bloc d'identité présent dans la moitié
haute.

Règles de classification :

- ENGAGEMENT_DOMICILIATION :
  le titre contient « ENGAGEMENT DE DOMICILIATION »
  ou « CONTRAT DES SALARIES ETRANGERS ».

- CONTRAT_TRAVAIL :
  le titre contient « CONTRAT DE TRAVAIL A DUREE DETERMINEE ».

- CONTRAT_SPECIFIQUE :
  le titre contient « CONTRAT DE TRAVAIL SPECIFIQUE
  A LA MAIN D'OEUVRE ETRANGERE ».

- TITRE_TRAVAIL :
  la page contient un bloc d'identité du travailleur, reconnaissable à
  AU MOINS DEUX des éléments suivants :
    - une photographie d'identité ;
    - les libellés « Nom » et « Prénom » suivis de valeurs ;
    - les libellés « Date de naissance » / « Lieu de naissance » ;
    - le libellé « Date d'entrée en Algérie » ;
    - des libellés arabes d'identité (اللقب، الإسم، تاريخ الإزدياد).
  Cette catégorie s'applique même si la page comporte aussi des cachets,
  un QR code, du texte de loi ou un second document en dessous.

- PERMIS_TRAVAIL_COUVERTURE :
  UNIQUEMENT si la page ne contient AUCUN bloc d'identité du travailleur
  et se limite au titre « Permis de Travail », au numéro de série et aux
  extraits de loi.

- AUTRE :
  aucun type ne correspond clairement.

Ne te base jamais uniquement sur le numéro de page.

Retourne uniquement ce JSON :
{
  "type_document": "TYPE",
  "confidence": 0.00,
  "titre_detecte": "TITRE BRUT OU null",
  "bloc_identite_present": true
}
"""

COMMON_RAW_RULES = """
Tu analyses une seule image, qui peut être une page entière ou un
recadrage d'une page.

RÈGLES OBLIGATOIRES :
1. Extraire uniquement les champs demandés.
2. Pour chaque champ, identifier d'abord le libellé et la structure du formulaire.
3. IMPORTANT — FORMULAIRES PRÉ-IMPRIMÉS :
   la couche des valeurs peut être décalée verticalement par rapport aux libellés.
   Le décalage peut être VERS LE HAUT ou VERS LE BAS et peut concerner TOUTE LA PAGE.
   Ne jamais associer une valeur à un champ uniquement parce qu'elle est exactement
   sur la même ligne horizontale que le libellé.
4. Utiliser conjointement :
   - le libellé ;
   - l'ordre des champs du formulaire ;
   - les libellés voisins au-dessus et au-dessous ;
   - le type de valeur attendu (date, lieu, nom, montant, référence...) ;
   - la cohérence globale de la page.
5. Un décalage global doit rester cohérent sur la page : ne pas déplacer
   arbitrairement une seule valeur d'un champ vers un autre.
6. Conserver la valeur exactement comme elle apparaît : espaces, ponctuation,
   séparateurs, format de date et format de montant.
7. Ne corrige pas l'orthographe.
8. Ne normalise pas les dates.
9. Ne normalise pas les montants.
10. Ne sépare pas automatiquement le nom et le prénom.
11. Ne complète pas une valeur partiellement lisible.
12. N'utilise aucune valeur provenant d'une autre page.
13. Si le libellé est absent, si la valeur est illisible ou si l'association
    libellé/valeur reste ambiguë malgré la structure de la page : retourne null.
14. N'invente jamais une valeur.
15. Retourne uniquement un objet JSON valide, sans commentaire.
16. Un champ absent du recadrage que tu analyses doit valoir null.
    Ne devine pas ce qui se trouve hors de l'image.
"""

PROMPT_ENGAGEMENT = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION

MISE EN PAGE : sur certains engagements, toutes les valeurs imprimées peuvent être
décalées verticalement de façon uniforme vers le haut ou vers le bas par rapport
aux libellés. Utilise la structure entière du formulaire et le type attendu de chaque
valeur ; ne fais jamais un appariement strict par ligne.

Extrais exactement les clés suivantes :

{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_ADRESSE_CLIENT": null,
  "DOM_AGENCE_DOMICILIATAIRE": null,
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null,
  "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": null,
  "DOM_ADRESSE_EMPLOYEUR": null,
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null,
  "DOM_DATE_SIGNATURE": null
}

Libellés et règles :

- DOM_NOM_RAISON_SOCIAL_CLIENT :
  valeur après « Nom et raison sociale ».
  La valeur peut être répartie en deux colonnes (nom puis prénom) :
  recopier les deux, séparés par un espace.

- DOM_COMPTE_LOCAL :
  valeur après « N de compte » ou « N° de compte ».

- DOM_ADRESSE_CLIENT :
  valeur après la première occurrence de « Adresse »
  dans la section « Identification du client ».

- DOM_AGENCE_DOMICILIATAIRE :
  valeur après « Agence domiciliataire », en haut à droite.

- DOM_NUMERO_CONTRAT :
  valeur après « Numéro du contrat ».

- DOM_DUREE_CONTRAT_MOIS :
  valeur après « Durée du contrat » ou « Duré du contrat ».

- DOM_DATE_DEBUT_CONTRAT :
  valeur après « Date de début de contrat ».

- DOM_DATE_FIN_CONTRAT :
  valeur après « Date de fin de contrat ».

- DOM_NOM_RAISON_SOCIAL_EMPLOYEUR :
  valeur après « Nom et raison sociale de L'Employeur ».

- DOM_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de L'Employeur ».
  Continuer sur la ligne suivante si l'adresse se poursuit.

- DOM_SALAIRE_NET_MENSUEL :
  valeur après « Salaire net mensuel ».

- DOM_PART_TRANSFERABLE :
  valeur après « Montant de la part transférable ».

- DOM_TAUX_TRANSFERABLE :
  valeur après « Pourcentage en regard du salaire net mensuel ».

- DOM_MONTANT_TOTAL_DOMICILIE :
  valeur après « Montant domicilié en DZD ».
  Ce champ est souvent laissé vide : retourner null dans ce cas.

- DOM_DATE_SIGNATURE :
  date manuscrite ou imprimée située près de la mention
  « lu et approuvé », en bas de page.
"""

PROMPT_CONTRAT = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_TRAVAIL

Extrais exactement les clés suivantes :

{
  "CTR_REFERENCE_DOCUMENT": null,
  "CTR_TYPE": null,
  "CTR_EMPLOYEUR": null,
  "CTR_ACTIVITE_EMPLOYEUR": null,
  "CTR_DUREE_MOIS": null,
  "CTR_DATE_DEBUT_CONTRAT": null,
  "CTR_POSTE": null,
  "CTR_NOM_PRENOM_TRAVAILLEUR": null,
  "CTR_PERE_NOM_PRENOM": null,
  "CTR_MERE_NOM_PRENOM": null,
  "CTR_NATIONALITE": null,
  "CTR_DATE_NAISSANCE": null,
  "CTR_LIEU_PAYS_NAISSANCE": null,
  "CTR_ADRESSE_ALGERIE": null,
  "CTR_QUALIFICATION": null,
  "CTR_NUMERO_PERMIS_TRAVAIL": null,
  "CTR_DATE_DELIVRANCE_PERMIS": null,
  "CTR_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTR_DATE_FIN_VALIDITE_PERMIS": null,
  "CTR_SALAIRE_BRUT": null,
  "CTR_SALAIRE_NET": null,
  "CTR_AFFILIATION_SS": null,
  "CTR_NUMERO_EMPLOYEUR": null,
  "CTR_DATE_SIGNATURE": null,
  "CTR_REFERENCE_DOMICILIATION": null,
  "CTR_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTR_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTR_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :

- CTR_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche,
  par exemple « TR-6166 ».

- CTR_TYPE :
  titre complet du document.

- CTR_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTR_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTR_DUREE_MOIS :
  valeur après « pour une durée de : »
  et avant « à compter du ».

- CTR_DATE_DEBUT_CONTRAT :
  valeur après « à compter du : ».

- CTR_POSTE :
  valeur après « En qualité de : ».

- CTR_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTR_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTR_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTR_NATIONALITE :
  valeur après « Nationalité : ».

- CTR_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTR_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTR_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTR_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTR_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTR_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTR_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « Valable du ».

- CTR_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

- CTR_SALAIRE_BRUT :
  valeur après « Montant du salaire mensuel brut : ».

- CTR_SALAIRE_NET :
  valeur après « Montant du salaire mensuel net : ».

- CTR_AFFILIATION_SS :
  valeur après « Affiliation à la sécurité sociale : ».

- CTR_NUMERO_EMPLOYEUR :
  valeur après « Employeur : ».

- CTR_DATE_SIGNATURE :
  date après « Fait à : Bethioua, le ».

- CTR_REFERENCE_DOMICILIATION :
  dans le cachet « DOMICILIATION IMPORT »,
  recopier les cinq cases dans l'ordre
  et les séparer par « | ».
  Exemple : 271901|2026.1|40|00119|DZD
  null si ce cachet est absent de la page.

Contrôles visuels :
- CTR_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature du Travailleur Etranger », sinon false.

- CTR_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature de l'Employeur », sinon false.

- CTR_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone
  « Signature de l'Employeur », sinon false.
"""

PROMPT_CONTRAT_SPECIFIQUE = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_SPECIFIQUE

Extrais exactement les clés suivantes :

{
  "CTS_REFERENCE_DOCUMENT": null,
  "CTS_SAP_ID": null,
  "CTS_EMPLOYEUR": null,
  "CTS_ACTIVITE_EMPLOYEUR": null,
  "CTS_DUREE_MOIS": null,
  "CTS_DATE_DEBUT_CONTRAT": null,
  "CTS_POSTE": null,
  "CTS_NOM_PRENOM_TRAVAILLEUR": null,
  "CTS_PERE_NOM_PRENOM": null,
  "CTS_MERE_NOM_PRENOM": null,
  "CTS_NATIONALITE": null,
  "CTS_DATE_NAISSANCE": null,
  "CTS_LIEU_PAYS_NAISSANCE": null,
  "CTS_ADRESSE_ALGERIE": null,
  "CTS_QUALIFICATION": null,
  "CTS_NUMERO_PERMIS_TRAVAIL": null,
  "CTS_DATE_DELIVRANCE_PERMIS": null,
  "CTS_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTS_DATE_FIN_VALIDITE_PERMIS": null,
  "CTS_LIGNE_SALAIRE_BRUTE": null,
  "CTS_SALAIRE_NET": null,
  "CTS_SALAIRE_NET_ANCIEN": null,
  "CTS_MENTION_AU_LIEU_DE_PRESENTE": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_NUMERO_SS_PAYS_ORIGINE": null,
  "CTS_NUMERO_SS_ALGERIE": null,
  "CTS_DATE_DOCUMENT": null,
  "CTS_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTS_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTS_CACHET_EMPLOYEUR_PRESENT": null,
  "CTS_VISA_INSPECTION_TRAVAIL_PRESENT": null
}

Libellés et règles :

- CTS_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche, par exemple « TR-6166 ».

- CTS_SAP_ID :
  valeur après « SAP id - » en haut de page.

- CTS_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTS_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTS_DUREE_MOIS :
  valeur après « pour une durée de : ».

- CTS_DATE_DEBUT_CONTRAT :
  valeur après « A compter du : ».

- CTS_POSTE :
  valeur après « en qualité de : ».

- CTS_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTS_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTS_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTS_NATIONALITE :
  valeur après « Nationalité : ».

- CTS_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTS_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTS_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTS_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTS_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTS_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTS_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « valable du ».

- CTS_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

--- LIGNE DE SALAIRE : RÈGLE PARTICULIÈRE ---

Cette ligne peut prendre DEUX formes :

  Forme A (salaire inchangé) :
    « Salaire mensuel de base net : 506,471.38 »

  Forme B (augmentation de salaire) :
    « Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29 »

- CTS_LIGNE_SALAIRE_BRUTE :
  recopier la ligne ENTIÈRE telle qu'elle apparaît, du libellé
  « Salaire mensuel de base net » jusqu'à la fin de la ligne,
  sans rien retirer.

- CTS_SALAIRE_NET :
  le PREMIER montant de cette ligne, c'est-à-dire celui situé
  immédiatement après « Salaire mensuel de base net : ».
  En forme B, c'est le NOUVEAU salaire, celui placé AVANT
  « au lieu de ». Ne jamais recopier « au lieu de » ni ce qui suit.

- CTS_SALAIRE_NET_ANCIEN :
  le SECOND montant de cette ligne, celui placé APRÈS
  « au lieu de ». C'est l'ANCIEN salaire.
  null si la mention « au lieu de » est absente de cette ligne.

- CTS_MENTION_AU_LIEU_DE_PRESENTE :
  true si la ligne de salaire contient « au lieu de », sinon false.

--- SUITE ---

- CTS_PART_TRANSFERABLE :
  valeur après « La part transférable : ».

- CTS_PART_PAYABLE_DZD :
  valeur après « La part payable en dinars algérien : ».

- CTS_NUMERO_SS_PAYS_ORIGINE :
  valeur après « Dans le pays d'origine : ».

- CTS_NUMERO_SS_ALGERIE :
  valeur après « En Algérie : ».

- CTS_DATE_DOCUMENT :
  date après « Fait à : Bethioua, le ».

Contrôles visuels :
- CTS_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature du Travailleur Etranger ».

- CTS_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature de l'Employeur ».

- CTS_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone employeur.

- CTS_VISA_INSPECTION_TRAVAIL_PRESENT :
  true si le bas de page comporte un cachet ou une mention manuscrite
  près de « Le présent contrat a été visé par nous ».
"""

PROMPT_TITRE_TRAVAIL = COMMON_RAW_RULES + """
TYPE ATTENDU : TITRE_TRAVAIL

MISE EN PAGE : sur certains titres/permis, toute la couche des valeurs peut être
décalée verticalement vers le haut OU vers le bas par rapport aux libellés.
Le décalage est global à la page. Pour Du/Au/Durée/Lieu de travail et les champs
d'identité, utilise l'ordre logique, le type attendu et la cohérence de l'ensemble.
Exemple : ORAN ne peut pas être une date de début ; une date ne doit pas être prise
comme lieu de travail. Si l'association reste ambiguë, retourne null.

DESCRIPTION DE LA PAGE :
Il s'agit d'un titre de travail algérien, bilingue arabe/français,
photocopié en noir et blanc. La qualité est dégradée, les valeurs sont
souvent inscrites sur des lignes de pointillés, et des cachets ronds
peuvent recouvrir partiellement le texte.

MISE EN PAGE — À LIRE ATTENTIVEMENT :
Le document est organisé en DEUX COLONNES.

  COLONNE DE GAUCHE — le poste et l'employeur.
  Chaque ligne suit le schéma :
      libellé français ..... valeur .....        libellé arabe
  Le libellé arabe est collé au bord DROIT de la colonne gauche.
  La valeur se trouve ENTRE le libellé français et le libellé arabe.
  Libellés : « Durée », « Du », « Au », « Lieu de travail »,
  « Nom de l'organisme employeur », « Adresse de l'organisme employeur »,
  « Fait à », « Le ».

  COLONNE DE DROITE — l'identité du travailleur, à côté de la photo.
  Chaque ligne suit le schéma :
      libellé français ..... valeur .....        libellé arabe
  Libellés : « Nom », « Prénom », « Date de naissance »,
  « Lieu de naissance », « Pays », « Nationalité », « Qualification »,
  « Date d'entrée en Algérie ».

RÈGLE CRITIQUE :
Ne JAMAIS confondre un libellé arabe avec une valeur.
La valeur est toujours le texte latin situé sur les pointillés,
entre le libellé français et le libellé arabe.
Si une ligne ne contient que des libellés et des pointillés vides,
retourner null pour ce champ.

Extrais exactement :

{
  "TTR_NUMERO_PERMIS": null,
  "TTR_NUMERO_MANUSCRIT": null,
  "TTR_POSTE": null,
  "TTR_DUREE": null,
  "TTR_DATE_DEBUT": null,
  "TTR_DATE_FIN": null,
  "TTR_LIEU_TRAVAIL": null,
  "TTR_EMPLOYEUR": null,
  "TTR_ADRESSE_EMPLOYEUR": null,
  "TTR_FAIT_A": null,
  "TTR_DATE_DELIVRANCE": null,
  "TTR_NOM": null,
  "TTR_PRENOM": null,
  "TTR_DATE_NAISSANCE": null,
  "TTR_LIEU_NAISSANCE": null,
  "TTR_PAYS": null,
  "TTR_NATIONALITE": null,
  "TTR_QUALIFICATION": null,
  "TTR_DATE_ENTREE_ALGERIE": null,
  "TTR_PHOTO_PRESENTE": null,
  "TTR_CACHET_PRESENT": null
}

Libellés et règles :

- TTR_NUMERO_PERMIS :
  référence encadrée en haut à gauche, de la forme
  « ( R ) 21-00002974 / 31-25-001448 ».
  Recopier la référence complète, y compris les deux parties
  séparées par « / ». Ignorer les parenthèses et la lettre isolée.

- TTR_NUMERO_MANUSCRIT :
  nombre manuscrit inscrit juste sous la référence encadrée,
  par exemple « 6466 ». null si absent.

RÈGLE V13.2 — BLOC STRUCTUREL PRIORITAIRE :
Avant d'associer POSTE / DUREE / DATE_DEBUT / DATE_FIN / LIEU_TRAVAIL,
observe ce bloc comme UNE STRUCTURE VERTICALE UNIQUE.

Les valeurs surimprimées peuvent être décalées globalement vers le haut ou
vers le bas par rapport aux libellés pré-imprimés. Plusieurs valeurs se
déplacent alors ensemble et leur ordre/espacement relatif reste généralement
stable. Ne fais PAS un appariement horizontal fixe libellé -> valeur.

Reconstruis d'abord :
POSTE -> DUREE -> DATE_DEBUT -> DATE_FIN -> LIEU_TRAVAIL.

POSTE peut occuper 1, 2 ou plusieurs lignes. Regroupe toutes ses lignes AVANT
de chercher DUREE. Une continuation de POSTE ne doit jamais devenir DUREE.
DUREE doit être sémantiquement une durée (AN/ANS/MOIS/JOUR...).
DATE_DEBUT est la première date après DUREE dans l'ordre vertical.
DATE_FIN est la deuxième date après DUREE.
LIEU_TRAVAIL vient après les deux dates et peut occuper plusieurs lignes.

En plus des 21 champs métier, retourne dans LE MÊME JSON cet objet d'audit :
"TTR_STRUCTURAL_BLOCK": {
  "poste_lines": [],
  "duree": null,
  "ordered_dates": [],
  "lieu_lines": [],
  "global_vertical_shift_suspected": false,
  "notes": []
}

Cet objet décrit uniquement ce qui est visible sur CETTE PAGE. N'utilise
aucune autre page et ne cherche pas à corriger la page par un autre document.

- TTR_POSTE :
  texte situé sous la phrase
  « Le titulaire du présent permis de travail est autorisé à occuper
  le poste de travail de ».
  Peut occuper une, deux ou plusieurs lignes : concaténer toutes les lignes.

- TTR_DUREE :
  première valeur sémantiquement compatible avec une durée après le poste,
  par exemple « 2 ANS, 0 JOURS » ou « 24 MOIS ».

RÈGLE STRUCTURELLE PRIORITAIRE POUR TTR_DATE_DEBUT / TTR_DATE_FIN :
  Sur ces titres de travail, les valeurs imprimées peuvent être décalées
  globalement vers le haut OU vers le bas par rapport aux libellés « Du » et « Au ».
  Pour éviter une mauvaise association, repère d'abord la zone verticale comprise
  ENTRE la valeur de « Durée » et le début de la valeur de « Lieu de travail ».

  Dans cette zone, lorsqu'il y a exactement deux dates plausibles de validité,
  lis-les DE HAUT EN BAS :
    - PREMIÈRE date rencontrée = TTR_DATE_DEBUT
    - SECONDE date rencontrée  = TTR_DATE_FIN

  Cette règle structurelle est prioritaire sur l'alignement horizontal avec
  les libellés « Du » et « Au ».

  Vérifie que TTR_DATE_DEBUT < TTR_DATE_FIN.
  Si TTR_DUREE est lisible, utilise-la comme preuve locale pour vérifier que
  l'intervalle est compatible avec la durée. Ce contrôle sert uniquement à
  valider l'association visuelle des lignes ; il ne modifie aucune valeur RAW.

  Si les dates restent ambiguës, ne devine pas.

- TTR_DATE_DEBUT :
  première date de validité rencontrée de haut en bas dans la zone
  située entre « Durée » et « Lieu de travail ».

- TTR_DATE_FIN :
  seconde date de validité rencontrée de haut en bas dans cette même zone.

- TTR_LIEU_TRAVAIL :
  valeur après « Lieu de travail ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_EMPLOYEUR :
  valeur après « Nom de l'organisme employeur ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de l'organisme employeur ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_FAIT_A :
  valeur après « Fait à ».

- TTR_DATE_DELIVRANCE :
  valeur après « Le », sous « Fait à ».
  Cette date est fréquemment recouverte par un cachet rond :
  si elle reste illisible, retourner null plutôt que de deviner.

- TTR_NOM :
  valeur après « Nom », ligne portant le libellé arabe « اللقب ».
  C'est le nom de famille seul.

- TTR_PRENOM :
  valeur après « Prénom », ligne portant le libellé arabe « الإسم ».

- TTR_DATE_NAISSANCE :
  valeur après « Date de naissance »,
  ligne portant le libellé arabe « تاريخ الإزدياد ».

- TTR_LIEU_NAISSANCE :
  valeur après « Lieu de naissance »,
  ligne portant le libellé arabe « مكان الإزدياد ».
  La valeur peut associer une ville et un pays séparés par « / » :
  recopier l'ensemble.

- TTR_PAYS :
  valeur après « Pays », ligne portant le libellé arabe « البلد ».

- TTR_NATIONALITE :
  valeur après « Nationalité »,
  ligne portant le libellé arabe « الجنسية ».

- TTR_QUALIFICATION :
  valeur après « Qualification »,
  ligne portant le libellé arabe « التأهيل ».
  La valeur peut être coupée en fin de ligne et se poursuivre sur la
  ligne suivante : recopier l'ensemble sans ajouter d'espace au point
  de coupure si le mot est manifestement scindé.

- TTR_DATE_ENTREE_ALGERIE :
  valeur après « Date d'entrée en Algérie »,
  ligne portant le libellé arabe « تاريخ الدخول إلى الجزائر ».

Contrôles visuels :
- TTR_PHOTO_PRESENTE :
  true si une photographie d'identité est visible en haut à droite.

- TTR_CACHET_PRESENT :
  true si au moins un cachet rond est visible sur le document.
"""

PROMPT_PERMIS_COUVERTURE = COMMON_RAW_RULES + """
TYPE ATTENDU : PERMIS_TRAVAIL_COUVERTURE

Cette image correspond à la couverture du permis de travail :
titre « جواز العمل / Permis de Travail », extraits de loi et cachet
de la Direction de l'Emploi de la Wilaya.

Extrais exactement :

{
  "PTR_NUMERO_SERIE": null,
  "PTR_WILAYA": null,
  "PTR_CACHET_DIRECTION_EMPLOI_PRESENT": null
}

- PTR_NUMERO_SERIE :
  numéro de série imprimé en bas de la couverture,
  après « N° de Série » ou isolé en bas à gauche.
  null si illisible.

- PTR_WILAYA :
  valeur après « Direction de l'Emploi de la Wilaya de : ».
  null si la ligne est vide.

- PTR_CACHET_DIRECTION_EMPLOI_PRESENT :
  true si un cachet officiel est visible sur cette zone.

Ne pas extraire le contenu des extraits de loi imprimés à droite.
"""

PROMPTS_EXTRACTION = {
    "ENGAGEMENT_DOMICILIATION": PROMPT_ENGAGEMENT,
    "CONTRAT_TRAVAIL": PROMPT_CONTRAT,
    "CONTRAT_SPECIFIQUE": PROMPT_CONTRAT_SPECIFIQUE,
    "TITRE_TRAVAIL": PROMPT_TITRE_TRAVAIL,
    "PERMIS_TRAVAIL_COUVERTURE": PROMPT_PERMIS_COUVERTURE,
}

TYPES_VALIDES = set(PROMPTS_EXTRACTION) | {"AUTRE"}


def champs_attendus_depuis_prompt(prompt):
    """
    Récupère la liste des clés depuis le squelette JSON du prompt.
    Évite de maintenir une seconde liste qui divergerait des prompts.
    """
    match = re.search(r"\{[^{}]*\}", prompt, flags=re.S)
    if not match:
        return []
    bloc = match.group(0)
    try:
        return list(json.loads(bloc).keys())
    except Exception:
        return re.findall(r'"([A-Z0-9_]+)"\s*:', bloc)



CHAMPS_ATTENDUS = {
    doc_type: champs_attendus_depuis_prompt(prompt)
    for doc_type, prompt in PROMPTS_EXTRACTION.items()
}

# ---------------------------------------------------------------------
# V13.5 — FORMAT DE SORTIE COMPACT
# ---------------------------------------------------------------------
# Qwen continue de recevoir toutes les règles métier et tous les noms de champs,
# mais il renvoie des clés courtes f01, f02... Python les remappe ensuite vers
# les noms canoniques. Le JSON RAW final conserve donc strictement les 99 champs.
COMPACT_ALIAS_BY_DOC = {
    dt: {field: f"f{i+1:02d}" for i,field in enumerate(fields)}
    for dt,fields in CHAMPS_ATTENDUS.items()
}
COMPACT_FIELD_BY_DOC = {
    dt: {alias: field for field,alias in mapping.items()}
    for dt,mapping in COMPACT_ALIAS_BY_DOC.items()
}

def _replace_first_json_skeleton_with_aliases(prompt, doc_type):
    mapping=COMPACT_ALIAS_BY_DOC.get(doc_type) or {}
    if not mapping:
        return prompt

    alias_skeleton={alias:None for alias in mapping.values()}
    match=re.search(r"\{[^{}]*\}", prompt, flags=re.S)
    if not match:
        return prompt

    alias_map_lines="\n".join(
        f"- {alias} = {field}"
        for field,alias in mapping.items()
    )
    compact_json=json.dumps(alias_skeleton,ensure_ascii=False,indent=2)

    result=prompt[:match.start()] + compact_json + prompt[match.end():]
    result += f"""

FORMAT DE SORTIE V13.5 — PRIORITÉ ABSOLUE :
Pour réduire la longueur de génération, retourne les champs métier avec les
ALIASES COURTS ci-dessous. Ne retourne PAS les noms longs des champs.

{alias_map_lines}

Retourne un seul objet JSON valide.
- Chaque alias ci-dessus doit être présent, avec sa valeur ou null.
- Pour TITRE_TRAVAIL uniquement, conserve aussi l'objet d'audit
  "TTR_STRUCTURAL_BLOCK" demandé plus haut.
- N'ajoute aucun commentaire ni texte hors JSON.
"""
    return result

def compact_prompt_for_doc(doc_type, prompt):
    if not COMPACT_OUTPUT_ENABLED:
        return prompt
    return _replace_first_json_skeleton_with_aliases(prompt,doc_type)

def expand_compact_extraction(parsed, doc_type):
    """Remappe f01/f02/... vers les champs canoniques, sans toucher aux métadonnées."""
    if not isinstance(parsed,dict):
        return {}
    reverse=COMPACT_FIELD_BY_DOC.get(doc_type) or {}
    if not reverse:
        return parsed

    out={}
    compact_seen=False
    for k,v in parsed.items():
        if k in reverse:
            out[reverse[k]]=v
            compact_seen=True
        else:
            # Conserver notamment TTR_STRUCTURAL_BLOCK.
            out[k]=v

    # Si Qwen a malgré tout utilisé les noms canoniques, ils restent acceptés.
    return out

# ---------------------------------------------------------------------
# V8.1 - champs critiques et stratégies de relecture
# ---------------------------------------------------------------------

CRITICAL_FIELDS = {
    'ENGAGEMENT_DOMICILIATION': {
        'DOM_NOM_RAISON_SOCIAL_CLIENT', 'DOM_COMPTE_LOCAL',
        'DOM_NUMERO_CONTRAT', 'DOM_DATE_DEBUT_CONTRAT',
        'DOM_DATE_FIN_CONTRAT', 'DOM_SALAIRE_NET_MENSUEL',
        'DOM_PART_TRANSFERABLE',
    },
    'CONTRAT_TRAVAIL': {
        'CTR_NOM_PRENOM_TRAVAILLEUR', 'CTR_NUMERO_PERMIS_TRAVAIL',
        'CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS',
        'CTR_SALAIRE_NET',
    },
    'CONTRAT_SPECIFIQUE': {
        'CTS_NOM_PRENOM_TRAVAILLEUR', 'CTS_NUMERO_PERMIS_TRAVAIL',
        'CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS',
        'CTS_SALAIRE_NET', 'CTS_PART_TRANSFERABLE',
    },
    'TITRE_TRAVAIL': {
        'TTR_NUMERO_PERMIS', 'TTR_DATE_DEBUT', 'TTR_DATE_FIN',
        'TTR_EMPLOYEUR', 'TTR_NOM', 'TTR_PRENOM',
        'TTR_DATE_NAISSANCE', 'TTR_NATIONALITE',
        'TTR_DATE_ENTREE_ALGERIE',
    },
    'PERMIS_TRAVAIL_COUVERTURE': {
        'PTR_NUMERO_SERIE',
    },
}

TITLE_IDENTITY_FIELDS = {
    'TTR_NOM', 'TTR_PRENOM', 'TTR_DATE_NAISSANCE',
    'TTR_LIEU_NAISSANCE', 'TTR_PAYS', 'TTR_NATIONALITE',
    'TTR_QUALIFICATION', 'TTR_DATE_ENTREE_ALGERIE',
    'TTR_PHOTO_PRESENTE', 'TTR_CACHET_PRESENT',
}

TITLE_POST_FIELDS = set(CHAMPS_ATTENDUS['TITRE_TRAVAIL']) - TITLE_IDENTITY_FIELDS

# Conservé comme documentation et pour les tests. Le moteur V8 construit
# ses jobs à partir de ces profils mais n'exécute plus toutes les stratégies
# séquentiellement par défaut.
STRATEGIES_EXTRACTION = {
    'TITRE_TRAVAIL': [
        {'nom': 'HD_BLOC_TITRE', 'crop_dynamique': 'titre'},
        {'nom': 'HD_COLONNE_IDENTITE', 'crop_dynamique': 'colonne_identite'},
        {'nom': 'HD_COLONNE_POSTE', 'crop_dynamique': 'colonne_poste'},
        {'nom': 'HD_SAFETY_TITRE', 'crop_dynamique': 'titre'},
    ],
    'PERMIS_TRAVAIL_COUVERTURE': [
        {'nom': 'HD_BLOC_COUVERTURE', 'crop_dynamique': 'couverture'},
        {'nom': 'PAGE_ENTIERE_HD'},
    ],
    'ENGAGEMENT_DOMICILIATION': [
        {'nom': 'STANDARD'}, {'nom': 'TARGETED_HD_FULL'},
    ],
    'CONTRAT_TRAVAIL': [
        {'nom': 'STANDARD'}, {'nom': 'TARGETED_HD_CENTRAL'},
    ],
    'CONTRAT_SPECIFIQUE': [
        {'nom': 'STANDARD'}, {'nom': 'TARGETED_HD_CENTRAL'},
    ],
}

# Indications courtes utilisées uniquement dans les mini-prompts de retry.
FIELD_HINTS = {
    # Engagement
    'DOM_NOM_RAISON_SOCIAL_CLIENT': 'valeur après « Nom et raison sociale » dans Identification du client',
    'DOM_COMPTE_LOCAL': 'valeur après « N de compte » ou « N° de compte »',
    'DOM_NUMERO_CONTRAT': 'valeur après « Numéro du contrat »',
    'DOM_DATE_DEBUT_CONTRAT': 'valeur après « Date de début de contrat »',
    'DOM_DATE_FIN_CONTRAT': 'valeur après « Date de fin de contrat »',
    'DOM_SALAIRE_NET_MENSUEL': 'valeur après « Salaire net mensuel »',
    'DOM_PART_TRANSFERABLE': 'valeur après « Montant de la part transférable »',
    # Contrat
    'CTR_NOM_PRENOM_TRAVAILLEUR': 'valeur après « A (Mr/Mme) »',
    'CTR_NUMERO_PERMIS_TRAVAIL': 'référence complète après « permis de travail N° »',
    'CTR_DATE_DEBUT_VALIDITE_PERMIS': 'première date après « Valable du »',
    'CTR_DATE_FIN_VALIDITE_PERMIS': 'date après « au » sur la ligne de validité',
    'CTR_SALAIRE_NET': 'valeur après « Montant du salaire mensuel net »',
    # Contrat spécifique
    'CTS_NOM_PRENOM_TRAVAILLEUR': 'valeur après « A (Mr/Mme) »',
    'CTS_NUMERO_PERMIS_TRAVAIL': 'référence complète après « permis de travail N° »',
    'CTS_DATE_DEBUT_VALIDITE_PERMIS': 'première date après « valable du »',
    'CTS_DATE_FIN_VALIDITE_PERMIS': 'date après « au » sur la ligne de validité',
    'CTS_SALAIRE_NET': 'PREMIER montant après « Salaire mensuel de base net », avant « au lieu de »',
    'CTS_SALAIRE_NET_ANCIEN': 'SECOND montant situé après « au lieu de »',
    'CTS_PART_TRANSFERABLE': 'valeur après « La part transférable »',
    # Titre de travail - colonne gauche / haut
    'TTR_NUMERO_PERMIS': 'référence encadrée en haut à gauche, complète avec les deux parties séparées par /',
    'TTR_NUMERO_MANUSCRIT': 'nombre manuscrit juste sous la référence encadrée',
    'TTR_POSTE': 'texte du poste sous la phrase « autorisé à occuper le poste de travail de »',
    'TTR_DUREE': 'valeur après « Durée »',
    'TTR_DATE_DEBUT': 'valeur après « Du »',
    'TTR_DATE_FIN': 'valeur après « Au »',
    'TTR_LIEU_TRAVAIL': 'valeur après « Lieu de travail »',
    'TTR_EMPLOYEUR': 'valeur après « Nom de l’organisme employeur »',
    'TTR_ADRESSE_EMPLOYEUR': 'valeur après « Adresse de l’organisme employeur »',
    'TTR_FAIT_A': 'valeur après « Fait à »',
    'TTR_DATE_DELIVRANCE': 'date après « Le » sous « Fait à »',
    # Titre de travail - colonne droite / identité
    'TTR_NOM': 'valeur latine après « Nom », avant le libellé arabe',
    'TTR_PRENOM': 'valeur latine après « Prénom », avant le libellé arabe',
    'TTR_DATE_NAISSANCE': 'valeur après « Date de naissance »',
    'TTR_LIEU_NAISSANCE': 'valeur après « Lieu de naissance »',
    'TTR_PAYS': 'valeur après « Pays »',
    'TTR_NATIONALITE': 'valeur après « Nationalité »',
    'TTR_QUALIFICATION': 'valeur après « Qualification »',
    'TTR_DATE_ENTREE_ALGERIE': 'valeur après « Date d’entrée en Algérie »',
    'TTR_PHOTO_PRESENTE': 'true si la photo d’identité est visible, sinon false',
    'TTR_CACHET_PRESENT': 'true si un cachet rond est visible, sinon false',
    # Couverture
    'PTR_NUMERO_SERIE': 'numéro après « N° de Série » sur la couverture Permis de Travail',
    'PTR_WILAYA': 'valeur après « Direction de l’Emploi de la Wilaya de »',
    'PTR_CACHET_DIRECTION_EMPLOI_PRESENT': 'true si le cachet officiel est visible',
}


def build_targeted_prompt(doc_type, fields):
    """Mini-prompt : uniquement les champs à relire/corriger."""
    fields = [f for f in fields if f in CHAMPS_ATTENDUS.get(doc_type, [])]
    skeleton = {f: None for f in fields}
    hints = []
    for field in fields:
        hint = FIELD_HINTS.get(field)
        if hint:
            hints.append(f'- {field}: {hint}.')
        else:
            readable = field.split('_', 1)[-1].replace('_', ' ').lower()
            hints.append(f'- {field}: rechercher le champ « {readable} » sur ce document.')

    return f"""
Analyse uniquement ce recadrage du document {doc_type}.

OBJECTIF : relire uniquement les champs listés ci-dessous.
- Recopie la valeur exactement comme elle apparaît.
- Ne normalise ni date ni montant.
- N'utilise aucune autre page.
- Si la valeur est absente ou illisible : null.
- N'invente jamais.
- Retourne uniquement un JSON valide avec exactement ces clés.

INDICES :
{chr(10).join(hints)}

JSON ATTENDU :
{json.dumps(skeleton, ensure_ascii=False, indent=2)}
"""


def critical_problems(record):
    doc_type = record.get('doc_type')
    data = record.get('raw_data') or {}
    problems = []
    for field in CRITICAL_FIELDS.get(doc_type, set()):
        value = data.get(field)
        if value in (None, '') or value_is_suspect(field, value):
            problems.append(field)
    return sorted(problems)




# ---------------------------------------------------------------------
# FINAL V12 — validation sémantique légère (sans normaliser le RAW)
# ---------------------------------------------------------------------
DATE_FIELDS = {
    'DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT','DOM_DATE_SIGNATURE',
    'CTR_DATE_DEBUT_CONTRAT','CTR_DATE_NAISSANCE','CTR_DATE_DELIVRANCE_PERMIS',
    'CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS','CTR_DATE_SIGNATURE',
    'CTS_DATE_DEBUT_CONTRAT','CTS_DATE_NAISSANCE','CTS_DATE_DELIVRANCE_PERMIS',
    'CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS','CTS_DATE_DOCUMENT',
    'TTR_DATE_DEBUT','TTR_DATE_FIN','TTR_DATE_DELIVRANCE','TTR_DATE_NAISSANCE',
    'TTR_DATE_ENTREE_ALGERIE',
}

AMOUNT_FIELDS = {
    'DOM_SALAIRE_NET_MENSUEL','DOM_PART_TRANSFERABLE','DOM_TAUX_TRANSFERABLE',
    'DOM_MONTANT_TOTAL_DOMICILIE','CTR_SALAIRE_BRUT','CTR_SALAIRE_NET',
    'CTS_LIGNE_SALAIRE_BRUTE','CTS_SALAIRE_NET','CTS_SALAIRE_NET_ANCIEN',
    'CTS_PART_TRANSFERABLE','CTS_PART_PAYABLE_DZD',
}

DURATION_FIELDS = {'DOM_DUREE_CONTRAT_MOIS','CTR_DUREE_MOIS','CTS_DUREE_MOIS','TTR_DUREE'}
BOOLEAN_FIELDS = {
    'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE','CTR_SIGNATURE_EMPLOYEUR_PRESENTE','CTR_CACHET_EMPLOYEUR_PRESENT',
    'CTS_MENTION_AU_LIEU_DE_PRESENTE','CTS_SIGNATURE_TRAVAILLEUR_PRESENTE',
    'CTS_SIGNATURE_EMPLOYEUR_PRESENTE','CTS_CACHET_EMPLOYEUR_PRESENT','CTS_VISA_INSPECTION_TRAVAIL_PRESENT',
    'TTR_PHOTO_PRESENTE','TTR_CACHET_PRESENT','PTR_CACHET_DIRECTION_EMPLOI_PRESENT',
}
PERMIT_REFERENCE_FIELDS = {'TTR_NUMERO_PERMIS','CTR_NUMERO_PERMIS_TRAVAIL','CTS_NUMERO_PERMIS_TRAVAIL','PTR_NUMERO_SERIE'}

# Pour ces champs, une valeur qui ressemble clairement à une date indique souvent
# un décalage de formulaire (ex. LIEU_TRAVAIL = 14/08/2025).
TEXT_FIELDS_REJECT_DATE = {
    'DOM_NOM_RAISON_SOCIAL_CLIENT','DOM_ADRESSE_CLIENT','DOM_AGENCE_DOMICILIATAIRE',
    'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR','DOM_ADRESSE_EMPLOYEUR',
    'CTR_EMPLOYEUR','CTR_ACTIVITE_EMPLOYEUR','CTR_POSTE','CTR_NOM_PRENOM_TRAVAILLEUR',
    'CTR_PERE_NOM_PRENOM','CTR_MERE_NOM_PRENOM','CTR_NATIONALITE','CTR_LIEU_PAYS_NAISSANCE',
    'CTR_ADRESSE_ALGERIE','CTR_QUALIFICATION',
    'CTS_EMPLOYEUR','CTS_ACTIVITE_EMPLOYEUR','CTS_POSTE','CTS_NOM_PRENOM_TRAVAILLEUR',
    'CTS_PERE_NOM_PRENOM','CTS_MERE_NOM_PRENOM','CTS_NATIONALITE','CTS_LIEU_PAYS_NAISSANCE',
    'CTS_ADRESSE_ALGERIE','CTS_QUALIFICATION',
    'TTR_POSTE','TTR_LIEU_TRAVAIL','TTR_EMPLOYEUR','TTR_ADRESSE_EMPLOYEUR','TTR_FAIT_A',
    'TTR_NOM','TTR_PRENOM','TTR_LIEU_NAISSANCE','TTR_PAYS','TTR_NATIONALITE','TTR_QUALIFICATION',
    'PTR_WILAYA',
}

_MONTH_WORDS = ('JANVIER','FEVRIER','FÉVRIER','MARS','AVRIL','MAI','JUIN','JUILLET',
                'AOUT','AOÛT','SEPTEMBRE','OCTOBRE','NOVEMBRE','DECEMBRE','DÉCEMBRE')


def _clean_scalar_text(value):
    return '' if value is None else str(value).strip()


def looks_like_date(value):
    s=_clean_scalar_text(value).upper()
    if not s:
        return False
    # Formats numériques usuels. On ne normalise rien ; on vérifie seulement la forme.
    if re.search(r'\b(?:0?[1-9]|[12]\d|3[01])[./-](?:0?[1-9]|1[0-2])[./-](?:19|20)\d{2}\b', s):
        return True
    if re.search(r'\b(?:19|20)\d{2}[./-](?:0?[1-9]|1[0-2])[./-](?:0?[1-9]|[12]\d|3[01])\b', s):
        return True
    if any(m in s for m in _MONTH_WORDS) and re.search(r'\b(?:19|20)\d{2}\b',s):
        return True
    return False


def parse_date_for_coherence(value):
    """Parse seulement pour contrôle de cohérence ; ne modifie jamais le RAW."""
    s=_clean_scalar_text(value)
    if not s:
        return None
    m=re.search(r'\b(\d{1,2})[./-](\d{1,2})[./-]((?:19|20)\d{2})\b',s)
    if m:
        try: return datetime(int(m.group(3)),int(m.group(2)),int(m.group(1)))
        except Exception: return None
    m=re.search(r'\b((?:19|20)\d{2})[./-](\d{1,2})[./-](\d{1,2})\b',s)
    if m:
        try: return datetime(int(m.group(1)),int(m.group(2)),int(m.group(3)))
        except Exception: return None
    return None


def semantic_issue_for_field(field,value):
    if is_missing_raw(value):
        return None
    s=_clean_scalar_text(value)
    digits=re.sub(r'\D','',s)

    if field in DATE_FIELDS and not looks_like_date(s):
        return 'EXPECTED_DATE'
    if field in AMOUNT_FIELDS and len(digits)==0:
        return 'EXPECTED_NUMERIC_VALUE'
    if field in DURATION_FIELDS:
        # Une durée doit contenir une unité temporelle explicite.
        # Evite le faux positif réel : "TAGE / LAMINOR 2" contient un chiffre
        # mais correspond à une continuation du poste, pas à une durée.
        dur=s.upper()
        if not re.search(r'\b\d{1,3}\s*(?:AN|ANS|ANNEE|ANNÉE|ANNEES|ANNÉES|MOIS|JOUR|JOURS)\b', dur):
            return 'EXPECTED_DURATION'
    if field == 'DOM_COMPTE_LOCAL' and len(digits) < 10:
        return 'EXPECTED_ACCOUNT_NUMBER'
    if field in PERMIT_REFERENCE_FIELDS and len(re.sub(r'[^A-Za-z0-9]','',s)) < 5:
        return 'EXPECTED_REFERENCE'
    if field in BOOLEAN_FIELDS:
        if isinstance(value,bool):
            return None
        if s.upper() not in {'TRUE','FALSE','VRAI','FAUX','OUI','NON','1','0'}:
            return 'EXPECTED_BOOLEAN'
    if field in TEXT_FIELDS_REJECT_DATE and looks_like_date(s):
        return 'EXPECTED_TEXT_GOT_DATE'
    return None


def coherence_issues_for_data(doc_type,data):
    """Contrôles prudents : uniquement contradictions évidentes."""
    issues=[]
    pairs={
        'ENGAGEMENT_DOMICILIATION': [('DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT','CONTRACT_DATE_ORDER')],
        'CONTRAT_TRAVAIL': [('CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS','PERMIT_DATE_ORDER')],
        'CONTRAT_SPECIFIQUE': [('CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS','PERMIT_DATE_ORDER')],
        'TITRE_TRAVAIL': [('TTR_DATE_DEBUT','TTR_DATE_FIN','WORK_PERMIT_DATE_ORDER')],
    }
    for f1,f2,code in pairs.get(doc_type,[]):
        d1=parse_date_for_coherence((data or {}).get(f1))
        d2=parse_date_for_coherence((data or {}).get(f2))
        if d1 and d2 and d1 > d2:
            issues.append({'code':code,'fields':[f1,f2],'detail':f'{f1}>{f2}'})

    # Cohérence durée TTR uniquement si les 3 éléments sont clairement interprétables.
    if doc_type=='TITRE_TRAVAIL':
        d1=parse_date_for_coherence((data or {}).get('TTR_DATE_DEBUT'))
        d2=parse_date_for_coherence((data or {}).get('TTR_DATE_FIN'))
        dur=_clean_scalar_text((data or {}).get('TTR_DUREE')).upper()
        m=re.search(r'\b(\d{1,2})\s*(?:AN|ANS|ANNEE|ANNÉE|ANNEES|ANNÉES)\b',dur)
        if d1 and d2 and m and d2>=d1:
            years=int(m.group(1))
            expected_days=years*365.25
            observed=(d2-d1).days
            # Tolérance large : uniquement incohérences manifestes.
            if years>0 and abs(observed-expected_days) > 120:
                issues.append({'code':'TTR_DURATION_DATE_INCONSISTENT',
                               'fields':['TTR_DUREE','TTR_DATE_DEBUT','TTR_DATE_FIN'],
                               'detail':f'duration={dur};days={observed}'})
    return issues


def assess_extraction_data(doc_type,data):
    expected=CHAMPS_ATTENDUS.get(doc_type) or []
    semantic=[]
    for f in expected:
        issue=semantic_issue_for_field(f,(data or {}).get(f))
        if issue:
            semantic.append({'field':f,'code':issue,'value':(data or {}).get(f)})
    coherence=coherence_issues_for_data(doc_type,data or {})
    critical_missing=[]
    for f in sorted(CRITICAL_FIELDS.get(doc_type,set())):
        v=(data or {}).get(f)
        if is_missing_raw(v):
            critical_missing.append(f)
    fill=taux_remplissage(data or {},expected)
    trigger=set(critical_missing)
    trigger.update(x['field'] for x in semantic)
    for x in coherence:
        trigger.update(x.get('fields') or [])
    low_fill=fill<SEUIL_REMPLISSAGE_MIN
    return {
        'critical_missing':critical_missing,
        'semantic_issues':semantic,
        'coherence_issues':coherence,
        'trigger_fields':sorted(trigger),
        'fill_rate':fill,
        'low_fill':bool(low_fill),
        'has_problem':bool(trigger or low_fill),
    }


def build_page_recovery_prompt(doc_type, issue_report, pass_no):
    problems=[]
    for f in issue_report.get('critical_missing') or []:
        problems.append(f'{f}: MISSING')
    for x in issue_report.get('semantic_issues') or []:
        problems.append(f"{x['field']}: {x['code']} (lu={x.get('value')!r})")
    for x in issue_report.get('coherence_issues') or []:
        problems.append(f"{x['code']}: {','.join(x.get('fields') or [])}")
    if issue_report.get('low_fill'):
        problems.append(f"LOW_FILL_RATE={issue_report.get('fill_rate')}")
    problem_text='\n'.join('- '+p for p in problems) or '- confirmation de la lecture précédente'

    return f"""
RECOVERY PAGE ENTIÈRE — PASS {pass_no}

Relis TOUTE la page de type {doc_type}. Ne relis aucune autre page du PDF.
La lecture précédente a produit au moins une donnée manquante, sémantiquement
incompatible ou incohérente, ou une correction doit être confirmée.

IMPORTANT LAYOUT : sur ces formulaires, la couche des valeurs peut être décalée
verticalement de façon globale VERS LE HAUT ou VERS LE BAS. Le même décalage
peut affecter tous les champs de la page. N'associe donc jamais une valeur à un
libellé uniquement par alignement horizontal. Utilise l'ordre du formulaire,
les champs voisins et le TYPE de valeur attendu.

Exemples de contradictions qui doivent être corrigées par relecture :
- un champ DATE contenant ORAN, INDE ou un poste ;
- un champ LIEU contenant une date ;
- date de début postérieure à date de fin ;
- valeurs décalées d'une ligne à cause de l'impression.

SI LE TYPE EST TITRE_TRAVAIL :
- TTR_DUREE doit être une vraie durée avec unité temporelle, par exemple
  « 2 ANS, 0 JOURS », « 1 AN » ou « 24 MOIS » ;
- une continuation du poste comme « TAGE / LAMINOR 2 » n'est PAS une durée ;
- si TTR_DUREE reste illisible, retourne null pour ce champ mais ne supprime
  jamais pour cette raison des dates début/fin correctement lues ;
- repère la zone comprise entre la valeur de « Durée » et la valeur de
  « Lieu de travail » ;
- les deux dates de validité présentes dans cette zone doivent être lues
  DE HAUT EN BAS ;
- première date = TTR_DATE_DEBUT ;
- seconde date = TTR_DATE_FIN ;
- ne te fie pas à l'alignement horizontal avec « Du » et « Au » ;
- confirme seulement que début < fin ;
- ne contrôle pas ici la compatibilité avec TTR_DUREE : ce contrôle appartient à la Partie 2 ;
- si cette structure reste ambiguë, retourne null plutôt que d'inventer.

PROBLÈMES / POINTS À CONFIRMER :
{problem_text}

Tu dois néanmoins réextraire TOUS les champs de cette page afin de disposer du
contexte complet. Ne permute jamais automatiquement des valeurs. Si une valeur
reste ambiguë, retourne null.

{PROMPTS_EXTRACTION[doc_type]}
"""

print('✅ Prompts V13 chargés — 99 champs inchangés + contrôles sémantiques/layout')
print('   Types gérés          :', ', '.join(sorted(PROMPTS_EXTRACTION)))
print('   Champs TITRE_TRAVAIL :', len(CHAMPS_ATTENDUS['TITRE_TRAVAIL']))
print('   Champs critiques TITRE:', len(CRITICAL_FIELDS['TITRE_TRAVAIL']))


## 8. Vérification du contrat de schéma


In [ ]:
FIELD_SCHEMA = {k:list(v) for k,v in CHAMPS_ATTENDUS.items()}
_current_hash = hashlib.sha256(
    json.dumps(FIELD_SCHEMA, sort_keys=True, ensure_ascii=False).encode('utf-8')
).hexdigest()
assert _current_hash == FIELD_SCHEMA_HASH, (_current_hash, FIELD_SCHEMA_HASH)
assert sum(len(v) for v in FIELD_SCHEMA.values()) == 99
print('✅ Schéma exact vérifié : 99 champs | hash', FIELD_SCHEMA_HASH[:16]+'…')
for k,v in FIELD_SCHEMA.items():
    print(f'   {k:28} : {len(v):2d} champs')


## 9. Checkpoints RAW V9


In [ ]:
def canonical_checkpoint_path(pdf_path):
    return JSON_DIR / f'{pdf_path.stem}.json'


def checkpoint_is_complete(dossier,pdf_path):
    if not isinstance(dossier,dict):
        return False
    if dossier.get('schema_version') != SCHEMA_VERSION:
        return False
    if dossier.get('pipeline_version') != PIPELINE_VERSION:
        return False
    if dossier.get('field_schema_hash') != FIELD_SCHEMA_HASH:
        return False
    if dossier.get('source_file') != pdf_path.name:
        return False
    if not dossier.get('page_records'):
        return False
    try:
        return dossier.get('source_sha256') == sha256_file(pdf_path)
    except Exception:
        return False


def load_existing_checkpoint(pdf_path):
    p=canonical_checkpoint_path(pdf_path)
    if not (RESUME and p.exists()):
        return None
    try:
        d=json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        return None
    return d if checkpoint_is_complete(d,pdf_path) else None


## 10. Moteur d’extraction RAW


In [ ]:

# =====================================================================
# V13.2 — STRUCTURAL-FIRST TTR
# =====================================================================
TTR_STRUCTURAL_FIELDS_V132=(
    "TTR_POSTE","TTR_DUREE","TTR_DATE_DEBUT","TTR_DATE_FIN","TTR_LIEU_TRAVAIL"
)

def _v132_join_lines(v):
    if not isinstance(v,list): return None
    p=[str(x).strip() for x in v if x is not None and str(x).strip()]
    return " ".join(p) if p else None

def _v132_duration_like(v):
    if v is None: return False
    return bool(re.search(
        r"\b\d{1,3}\s*(?:AN|ANS|ANNEE|ANNEES|ANNÉE|ANNÉES|MOIS|JOUR|JOURS)\b",
        str(v).upper()
    ))

def _v132_read_block(parsed):
    if not isinstance(parsed,dict) or not isinstance(parsed.get("TTR_STRUCTURAL_BLOCK"),dict):
        return None
    b=parsed["TTR_STRUCTURAL_BLOCK"]
    dates=b.get("ordered_dates") or []
    if not isinstance(dates,list): dates=[]
    dates=[str(x).strip() for x in dates if x is not None and str(x).strip()]
    return {
      "TTR_POSTE":_v132_join_lines(b.get("poste_lines")),
      "TTR_DUREE":str(b.get("duree")).strip() if b.get("duree") is not None and str(b.get("duree")).strip() else None,
      "TTR_DATE_DEBUT":dates[0] if len(dates)>0 else None,
      "TTR_DATE_FIN":dates[1] if len(dates)>1 else None,
      "TTR_LIEU_TRAVAIL":_v132_join_lines(b.get("lieu_lines")),
      "global_vertical_shift_suspected":bool(b.get("global_vertical_shift_suspected",False)),
      "notes":b.get("notes") if isinstance(b.get("notes"),list) else [],
    }

def _v132_validate_block(v):
    why=[]
    if not v: return False,["STRUCTURAL_BLOCK_MISSING"]
    if not v.get("TTR_POSTE"): why.append("POSTE_MISSING")
    if not _v132_duration_like(v.get("TTR_DUREE")): why.append("DURATION_NOT_RECOGNIZED")
    a,b=v.get("TTR_DATE_DEBUT"),v.get("TTR_DATE_FIN")
    d1,d2=parse_date_for_coherence(a),parse_date_for_coherence(b)
    if not (d1 and d2): why.append("TWO_VALID_DATES_NOT_FOUND")
    elif d1>=d2: why.append("DATE_ORDER_INVALID")
    elif not _pair_duration_compatible(a,b,v.get("TTR_DUREE")): why.append("DURATION_DATE_MISMATCH")
    if not v.get("TTR_LIEU_TRAVAIL"): why.append("LOCATION_MISSING")
    return not why,why

def _v132_apply_initial_block(record,parsed):
    if record.get("doc_type")!="TITRE_TRAVAIL": return False
    v=_v132_read_block(parsed)
    ok,why=_v132_validate_block(v)
    record["ttr_initial_structural_block"]=v
    record["ttr_initial_structural_block_valid"]=bool(ok)
    record["ttr_initial_structural_block_reasons"]=why
    record["ttr_structural_lock"]=False
    if not ok: return False

    for f in TTR_STRUCTURAL_FIELDS_V132:
        val=v.get(f)
        if not is_missing_raw(val):
            record["raw_data"][f]=val
            record.setdefault("field_candidates",{}).setdefault(f,[]).append({
              "value":val,"strategy":"TTR_INITIAL_STRUCTURAL_BLOCK",
              "semantic_valid":semantic_issue_for_field(f,val) is None,
              "semantic_issue":semantic_issue_for_field(f,val),
              "structural_authority":True
            })
    record["ttr_structural_lock"]=True
    record["ttr_structural_lock_fields"]=list(TTR_STRUCTURAL_FIELDS_V132)
    record["ttr_structural_date_resolution"]="RESOLVED_BY_INITIAL_STRUCTURAL_BLOCK"
    record["structural_confirmation_required"]=False
    record.setdefault("quality_flags",[]).append("TTR_INITIAL_STRUCTURAL_BLOCK_VALID")
    if v.get("global_vertical_shift_suspected"):
        record.setdefault("quality_flags",[]).append("TTR_GLOBAL_VERTICAL_SHIFT_SUSPECTED")
    return True

def _v132_locked(record,field):
    return bool(record.get("ttr_structural_lock")) and field in set(record.get("ttr_structural_lock_fields") or [])


# =====================================================================
# V13.3 — FAST-SAFE TTR
# =====================================================================
def _v133_initial_raw_ttr_block(record):
    raw=record.get("raw_data") or {}
    return {f:raw.get(f) for f in TTR_STRUCTURAL_FIELDS_V132}

def _v133_validate_initial_raw_ttr(record):
    """Valide sans nouvel appel Qwen le bloc TTR du premier passage."""
    v=_v133_initial_raw_ttr_block(record)
    reasons=[]
    poste=v.get("TTR_POSTE")
    duree=v.get("TTR_DUREE")
    debut=v.get("TTR_DATE_DEBUT")
    fin=v.get("TTR_DATE_FIN")
    lieu=v.get("TTR_LIEU_TRAVAIL")

    if is_missing_raw(poste):
        reasons.append("POSTE_MISSING")
    elif looks_like_date(poste) or _v132_duration_like(poste):
        reasons.append("POSTE_WRONG_SEMANTIC_TYPE")

    if not _v132_duration_like(duree):
        reasons.append("DURATION_NOT_RECOGNIZED")

    d1=parse_date_for_coherence(debut)
    d2=parse_date_for_coherence(fin)
    if not (d1 and d2):
        reasons.append("TWO_VALID_DATES_NOT_FOUND")
    elif d1>=d2:
        reasons.append("DATE_ORDER_INVALID")
    elif not _pair_duration_compatible(debut,fin,duree):
        reasons.append("DURATION_DATE_MISMATCH")

    if is_missing_raw(lieu):
        reasons.append("LOCATION_MISSING")
    elif re.match(r"^\s*\d{1,2}\s*[/.\-]\s*\d{1,2}\s*[/.\-]\s*\d{2,4}\b",str(lieu)):
        reasons.append("LOCATION_STARTS_WITH_DATE")

    for f in TTR_STRUCTURAL_FIELDS_V132:
        issue=semantic_issue_for_field(f,v.get(f))
        if issue is not None:
            reasons.append(f"{f}:{issue}")

    return len(reasons)==0,list(dict.fromkeys(reasons)),v

def _v133_fast_safe_lock_from_initial_fields(record):
    """
    Si l'objet structurel V13.2 est absent mais que les 5 champs du premier
    passage sont déjà strictement cohérents, on les verrouille et on évite
    l'observation structurelle + les recoveries inutiles.
    """
    if not FAST_SAFE_TTR_INITIAL_LOCK or record.get("doc_type")!="TITRE_TRAVAIL":
        return False

    if record.get("ttr_structural_lock"):
        record["ttr_fast_safe_status"]="LOCKED_FROM_INITIAL_STRUCTURAL_OBJECT"
        return True

    ok,reasons,v=_v133_validate_initial_raw_ttr(record)
    record["ttr_fast_safe_initial_validation_reasons"]=reasons
    if not ok:
        record["ttr_fast_safe_status"]="FALLBACK_V13_2_REQUIRED"
        return False

    record["ttr_initial_structural_block"]=dict(v)
    record["ttr_initial_structural_block_valid"]=True
    record["ttr_initial_structural_block_reasons"]=[]
    record["ttr_structural_lock"]=True
    record["ttr_structural_lock_fields"]=list(TTR_STRUCTURAL_FIELDS_V132)
    record["ttr_structural_date_resolution"]="RESOLVED_BY_INITIAL_FIELDS_FAST_SAFE"
    record["structural_confirmation_required"]=False
    record["ttr_fast_safe_status"]="LOCKED_FROM_INITIAL_FIELDS"
    record.setdefault("quality_flags",[]).append("TTR_FAST_SAFE_INITIAL_BLOCK_VALID")
    return True


# =====================================================================
# V13.4 — SPECIALIZED OBSERVATION LOCK
# =====================================================================
def _v134_lock_from_specialized_observation(record):
    """
    Après le fallback TTR_STRUCTURAL_OBSERVATION, exploite immédiatement
    cette lecture spécialisée au lieu de lancer encore un recovery générique.

    L'observation spécialisée est acceptée seulement si :
    - durée sémantiquement valide ;
    - exactement au moins deux dates ordonnées, début < fin ;
    - durée compatible avec les deux dates ;
    - lieu présent ;
    - poste présent dans l'observation, ou poste initial déjà sémantiquement valide.
    """
    if record.get("doc_type")!="TITRE_TRAVAIL" or record.get("ttr_structural_lock"):
        return bool(record.get("ttr_structural_lock"))

    obs_list=[
        o for o in (record.get("visual_observations") or [])
        if o.get("strategy")=="TTR_STRUCTURAL_OBSERVATION"
    ]
    if not obs_list:
        return False
    o=obs_list[-1]

    dates=o.get("ordered_dates_between_duration_and_location") or []
    if len(dates)<2:
        return False

    start,end=dates[0],dates[1]
    duration=o.get("duration_text")
    location=o.get("location_text")
    post=o.get("post_text")

    if not _v132_duration_like(duration):
        return False
    d1,d2=parse_date_for_coherence(start),parse_date_for_coherence(end)
    if not (d1 and d2 and d1<d2):
        return False
    if not _pair_duration_compatible(start,end,duration):
        return False
    if is_missing_raw(location):
        return False

    # Si le prompt spécialisé n'a pas relu le poste, garder le poste initial
    # seulement s'il est sémantiquement acceptable.
    if is_missing_raw(post):
        post=(record.get("raw_data") or {}).get("TTR_POSTE")
    if is_missing_raw(post) or semantic_issue_for_field("TTR_POSTE",post) is not None:
        return False

    authoritative={
        "TTR_POSTE":post,
        "TTR_DUREE":duration,
        "TTR_DATE_DEBUT":start,
        "TTR_DATE_FIN":end,
        "TTR_LIEU_TRAVAIL":location,
    }
    for f,v in authoritative.items():
        record["raw_data"][f]=v
        record.setdefault("field_candidates",{}).setdefault(f,[]).append({
            "value":v,
            "strategy":"TTR_STRUCTURAL_OBSERVATION_V134_LOCK",
            "semantic_valid":semantic_issue_for_field(f,v) is None,
            "semantic_issue":semantic_issue_for_field(f,v),
            "structural_authority":True,
        })

    record["ttr_initial_structural_block"]=dict(authoritative)
    record["ttr_initial_structural_block_valid"]=True
    record["ttr_initial_structural_block_reasons"]=[]
    record["ttr_structural_lock"]=True
    record["ttr_structural_lock_fields"]=list(TTR_STRUCTURAL_FIELDS_V132)
    record["ttr_structural_date_resolution"]="RESOLVED_BY_SPECIALIZED_OBSERVATION_V134"
    record["structural_confirmation_required"]=False
    record["ttr_fast_safe_status"]="LOCKED_FROM_SPECIALIZED_OBSERVATION"
    record.setdefault("quality_flags",[]).append("TTR_SPECIALIZED_OBSERVATION_LOCK_VALID")
    return True

def _v134_need_second_recovery(report, doc_type, changed_fields):
    """
    Le 2400 est réservé aux problèmes à impact réglementaire :
    - champ critique toujours manquant / invalide ;
    - incohérence inter-champs toujours présente ;
    - correction d'un champ critique au passage 2000.

    Une anomalie sémantique uniquement NON critique reste tracée pour la
    validation, mais ne justifie plus à elle seule un troisième appel Qwen.
    """
    critical=set(CRITICAL_FIELDS.get(doc_type,set()))
    unresolved_critical=set(report.get("critical_missing") or [])
    unresolved_critical.update(
        x.get("field") for x in (report.get("semantic_issues") or [])
        if isinstance(x,dict) and x.get("field") in critical
    )
    coherence=report.get("coherence_issues") or []
    critical_changed=set(changed_fields or []) & critical
    return bool(unresolved_critical or coherence or critical_changed)


In [ ]:

# =====================================================================
# V13 — EVIDENCE FIRST / OBSERVATION STRUCTURELLE
# =====================================================================
# Principe :
# 1) le VLM observe d'abord ce qui est visuellement présent ;
# 2) Python associe ensuite les observations à la structure du formulaire ;
# 3) les autres documents ne corrigent JAMAIS cette page en Partie 1.
#
# Pour le TTR, cette passe est volontairement indépendante de l'extraction
# champ-par-champ. Elle sert à détecter les décalages verticaux de formulaire.

TTR_OBSERVATION_PROMPT = r"""
Tu analyses UNE page de titre/permis de travail.

OBJECTIF : observer uniquement le bloc structurel :
POSTE -> DUREE -> DATE_DEBUT -> DATE_FIN -> LIEU_TRAVAIL.

Le formulaire peut avoir un décalage vertical GLOBAL vers le haut ou vers le bas.
Les valeurs se déplacent ensemble ; ne fais jamais un appariement horizontal fixe.

Règles :
1. POSTE peut avoir une ou plusieurs lignes : regroupe toutes ses lignes.
2. DUREE doit être une vraie durée (AN/ANS/MOIS/JOUR...).
3. Entre DUREE et LIEU, relève toutes les dates visibles dans l'ordre vertical.
4. Première date = début ; deuxième date = fin.
5. LIEU vient après les deux dates et peut avoir plusieurs lignes.
6. Si une valeur reste ambiguë : null. N'invente rien.

Retourne UNIQUEMENT :
{
  "post_text": null,
  "duration_text": null,
  "ordered_dates_between_duration_and_location": [],
  "location_text": null,
  "observation_notes": []
}
"""

def _ttr_observation_job(record, pdf_path):
    # Même zone TTR HD que l'extraction initiale, mais prompt indépendant.
    crops = crops_planche_permis(record['image'])
    record['frontiere_planche'] = crops['frontiere']
    return {
        'record': record,
        'prompt': TTR_OBSERVATION_PROMPT,
        'image': _render_for_strategy(
            record, pdf_path, 'TTR_STRUCTURAL_OBSERVATION',
            crops['titre'], IMAGE_MAX_SIZE_HAUTE_DEF
        ),
        'strategy': 'TTR_STRUCTURAL_OBSERVATION',
        'profile': 'HD',
        'mode': 'STRUCTURAL_OBSERVATION',
        'max_new_tokens': MAX_NEW_TOKENS_TTR_STRUCTURAL,
    }

def _merge_ttr_observation(job, output):
    r = job['record']
    parsed = parse_json_response(output.get('text',''))
    dates = parsed.get('ordered_dates_between_duration_and_location')
    if not isinstance(dates, list):
        dates = []
    dates = [x for x in dates if looks_like_date(x)]

    obs = {
        'strategy': job['strategy'],
        'post_text': parsed.get('post_text'),
        'duration_text': parsed.get('duration_text'),
        'ordered_dates_between_duration_and_location': dates,
        'location_text': parsed.get('location_text'),
        'permit_number': parsed.get('permit_number'),
        'surname': parsed.get('surname'),
        'given_name': parsed.get('given_name'),
        'birth_date': parsed.get('birth_date'),
        'entry_date_algeria': parsed.get('entry_date_algeria'),
        'observation_notes': parsed.get('observation_notes') or [],
        'tokens_in': int(output.get('tokens_in',0) or 0),
        'tokens_out': int(output.get('tokens_out',0) or 0),
        'elapsed_s': float(output.get('elapsed_s',0) or 0),
    }
    r.setdefault('visual_observations', []).append(obs)
    r['structural_observation_call_count']=int(
        r.get('structural_observation_call_count',0) or 0
    )+1

    # Candidat structurel : première date = début, seconde = fin.
    if len(dates) >= 2:
        d1, d2 = dates[0], dates[1]
        p1, p2 = parse_date_for_coherence(d1), parse_date_for_coherence(d2)
        if p1 and p2 and p1 < p2:
            r['ttr_observed_date_pair'] = {
                'TTR_DATE_DEBUT': d1,
                'TTR_DATE_FIN': d2,
                'method': 'VISUAL_ORDER_BETWEEN_DURATION_AND_LOCATION'
            }
            _add_field_candidate(r, 'TTR_DATE_DEBUT', d1, 'TTR_STRUCTURAL_OBSERVATION')
            _add_field_candidate(r, 'TTR_DATE_FIN', d2, 'TTR_STRUCTURAL_OBSERVATION')

            current_s = (r.get('raw_data') or {}).get('TTR_DATE_DEBUT')
            current_e = (r.get('raw_data') or {}).get('TTR_DATE_FIN')
            if (_candidate_key(current_s) != _candidate_key(d1) or
                _candidate_key(current_e) != _candidate_key(d2)):
                # Désaccord entre deux lectures indépendantes de la même page :
                # on exige une confirmation pleine page. Pas de correction silencieuse.
                r['structural_confirmation_required'] = True
                r.setdefault('quality_flags', []).append(
                    'TTR_DIRECT_VS_STRUCTURAL_OBSERVATION_DISAGREEMENT'
                )

    r['extraction_tokens_in'] += obs['tokens_in']
    r['extraction_tokens_out'] += obs['tokens_out']
    r['extraction_elapsed_s'] = round(
        r.get('extraction_elapsed_s',0) + obs['elapsed_s'], 3
    )

def run_ttr_structural_observations(records, pdf_path):
    jobs = [
        _ttr_observation_job(r, pdf_path)
        for r in records
        if r.get('doc_type') == 'TITRE_TRAVAIL'
        and not r.get('classification_review_required')
        and not r.get('ttr_structural_lock')
    ]
    for start in range(0, len(jobs), GPU_BATCH_SIZE_EXTRACTION_HD):
        chunk = jobs[start:start+GPU_BATCH_SIZE_EXTRACTION_HD]
        if not chunk:
            continue
        _psync(); _bt=_pnow()
        outs = ask_batch_mixed(
            [j['prompt'] for j in chunk],
            [j['image'] for j in chunk],
            max(j['max_new_tokens'] for j in chunk)
        )
        _psync(); _batch_s=_pnow()-_bt
        for j,o in zip(chunk, outs):
            _merge_ttr_observation(j,o)
            _r=j['record']
            page_pevent(
                _r.get('page_num'),_r.get('doc_type'),
                'TTR_STRUCTURAL','TTR_STRUCTURAL_OBSERVATION',
                _batch_s,len(chunk),
                o.get('tokens_in',0),o.get('tokens_out',0),
                note='TEMPS_BATCH_PARTAGE' if len(chunk)>1 else 'TEMPS_PAGE_DIRECT'
            )


In [ ]:

# =====================================================================
# V13.7.1 — TTR DATE / DURATION HELPERS
# Correctif autonome : toutes les dépendances du contrôle structurel TTR
# sont définies ici avant leur première utilisation.
# =====================================================================
import re
from datetime import datetime

def parse_date_flexible(value):
    """Retourne datetime ou None. Aucune correction OCR lettre->chiffre."""
    if value is None:
        return None
    s = str(value).strip()
    if not s:
        return None

    # Normalisation prudente des séparateurs seulement.
    s = re.sub(r"\s+", "", s)
    s = s.replace("-", "/").replace(".", "/")

    for fmt in ("%d/%m/%Y", "%d/%m/%y", "%Y/%m/%d"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            pass
    return None


def _duration_months_approx(duration_text):
    """Convertit une durée textuelle explicite en nombre approximatif de mois.
    Ex: '2 ANS, 0 JOURS' -> 24. Retourne None si non interprétable.
    """
    if duration_text is None:
        return None

    s = str(duration_text).upper().strip()
    if not s:
        return None

    years = months = days = 0
    found = False

    m = re.search(r"(\d+)\s*(?:AN|ANS|ANNEE|ANNEES|ANNÉE|ANNÉES)\b", s)
    if m:
        years = int(m.group(1))
        found = True

    m = re.search(r"(\d+)\s*(?:MOIS)\b", s)
    if m:
        months = int(m.group(1))
        found = True

    m = re.search(r"(\d+)\s*(?:JOUR|JOURS)\b", s)
    if m:
        days = int(m.group(1))
        found = True

    if not found:
        return None

    return years * 12 + months + days / 30.4375


def _pair_duration_compatible(date_start, date_end, duration_text, tolerance_days=35):
    """Vérifie start < end et, si la durée est lisible, sa cohérence avec l'intervalle.

    Si la durée n'est pas interprétable, une paire chronologique valide reste
    acceptable : on ne détruit jamais deux dates cohérentes uniquement parce
    que le champ durée est illisible.
    """
    d1 = parse_date_flexible(date_start)
    d2 = parse_date_flexible(date_end)

    if d1 is None or d2 is None or d1 >= d2:
        return False

    months = _duration_months_approx(duration_text)
    if months is None:
        return True

    observed_days = (d2 - d1).days
    expected_days = months * 30.4375

    return abs(observed_days - expected_days) <= float(tolerance_days)


# Régressions fictives, sans données réelles.
assert parse_date_flexible("14/04/2025") is not None
assert parse_date_flexible("texte") is None
assert _pair_duration_compatible("14/04/2025", "13/04/2027", "2 ANS, 0 JOURS")
assert not _pair_duration_compatible("13/04/2027", "14/04/2025", "2 ANS, 0 JOURS")
assert _pair_duration_compatible("14/04/2025", "13/04/2027", None)

print("✅ V13.7.1 helpers TTR chargés : dates + durée + cohérence")


In [ ]:
def _new_record_from_page(page, parsed, output):
    doc_type=parsed.get('type_document') or parsed.get('type') or 'AUTRE'
    try:
        confidence=float(parsed.get('confidence',0) or 0)
    except Exception:
        confidence=0.0
    bloc_identite=bool(parsed.get('bloc_identite_present'))
    if doc_type not in TYPES_VALIDES:
        doc_type='AUTRE'
    requalifie=False
    if bloc_identite and doc_type in ('PERMIS_TRAVAIL_COUVERTURE','AUTRE'):
        doc_type='TITRE_TRAVAIL'; confidence=max(confidence,CLASSIFICATION_THRESHOLD); requalifie=True
    if confidence<CLASSIFICATION_THRESHOLD and not requalifie:
        doc_type='AUTRE'
    return {
        'page_num':page['page_num'],'width':page['width'],'height':page['height'],
        'white_ratio':page['white_ratio'],'image':page['image'],
        'doc_type':doc_type,'titre_detecte':parsed.get('titre_detecte'),
        'bloc_identite_present':bloc_identite,'classification_requalifiee':requalifie,
        'classification_retry_fullres':False,'classification_confidence':confidence,
        'classification_raw_text':output.get('text'),
        'classification_attempts':[{
            'strategy':'LOWRES_1100','raw_text':output.get('text'),
            'parsed':parsed,'tokens_in':output.get('tokens_in',0),
            'tokens_out':output.get('tokens_out',0),'elapsed_s':output.get('elapsed_s',0),
        }],
        'classification_tokens_in':int(output.get('tokens_in',0) or 0),
        'classification_tokens_out':int(output.get('tokens_out',0) or 0),
        'classification_elapsed_s':float(output.get('elapsed_s',0) or 0),
        'raw_data':{},'extraction_status':'NON_LANCEE','extraction_error':None,
        'extraction_raw_text':None,'extraction_attempts':[],
        'extraction_strategies':[],'extraction_taux_remplissage':0.0,
        'extraction_call_count':0,'retry_call_count':0,
        'extraction_tokens_in':0,'extraction_tokens_out':0,'extraction_elapsed_s':0.0,
        'field_revisions':[],'critical_fields_missing':[],'quality_flags':[],
        'field_candidates':{},'field_disagreements':[],'recovery_history':[],
        'semantic_issues_initial':[],'semantic_issues_final':[],'coherence_issues_final':[],
        'page_recovery_triggered':False,'recovery_passes':0,
        'field_confidence':{},'extraction_confidence_score':0.0,
        'extraction_confidence_band':'LOW','confidence_method':CONFIDENCE_METHOD,
        'is_virtual_subdocument':False,
    }


def classify_pages(pages):
    records=[]
    for start in range(0,len(pages),GPU_BATCH_SIZE_CLASSIFICATION):
        batch=pages[start:start+GPU_BATCH_SIZE_CLASSIFICATION]
        outs=ask_batch(PROMPT_CLASSIFICATION,[image_for_classification(x['image']) for x in batch],MAX_NEW_TOKENS_CLASSIFICATION)
        _shared_cls=sum(float(o.get('elapsed_s',0) or 0) for o in outs)
        for page,out in zip(batch,outs):
            _rec=_new_record_from_page(page,parse_json_response(out['text']),out)
            records.append(_rec)
            page_pevent(
                page.get('page_num'), _rec.get('doc_type'),
                'CLASSIFICATION','CLASSIFICATION_BATCH',
                _shared_cls,len(batch),
                out.get('tokens_in',0),out.get('tokens_out',0)
            )

    if CLASSIFICATION_RETRY_ON_AUTRE or CLASSIFICATION_RETRY_LOW_CONFIDENCE:
        retry=[r for r in records if (r['doc_type']=='AUTRE' or float(r.get('classification_confidence',0) or 0) < CLASSIFICATION_HARD_MIN_CONFIDENCE)]
        for start in range(0,len(retry),GPU_BATCH_SIZE_CLASSIFICATION):
            batch=retry[start:start+GPU_BATCH_SIZE_CLASSIFICATION]
            outs=ask_batch(PROMPT_CLASSIFICATION,[r['image'] for r in batch],MAX_NEW_TOKENS_CLASSIFICATION)
            _shared_cls_retry=sum(float(o.get('elapsed_s',0) or 0) for o in outs)
            for r,out in zip(batch,outs):
                parsed=parse_json_response(out['text'])
                page_pevent(
                    r.get('page_num'), parsed.get('doc_type') or r.get('doc_type'),
                    'CLASSIFICATION_RETRY','FULLRES_1400_RETRY',
                    _shared_cls_retry,len(batch),
                    out.get('tokens_in',0),out.get('tokens_out',0)
                )
                attempt={'strategy':'FULLRES_1400_RETRY','raw_text':out.get('text'),'parsed':parsed,
                         'tokens_in':out.get('tokens_in',0),'tokens_out':out.get('tokens_out',0),'elapsed_s':out.get('elapsed_s',0)}
                new=_new_record_from_page({'page_num':r['page_num'],'width':r['width'],'height':r['height'],
                                           'white_ratio':r['white_ratio'],'image':r['image']},parsed,out)
                attempts=list(r.get('classification_attempts') or [])+[attempt]
                new['classification_attempts']=attempts
                new['classification_tokens_in']=sum(int(a.get('tokens_in',0) or 0) for a in attempts)
                new['classification_tokens_out']=sum(int(a.get('tokens_out',0) or 0) for a in attempts)
                new['classification_elapsed_s']=round(sum(float(a.get('elapsed_s',0) or 0) for a in attempts),3)
                new['classification_raw_text']='\n\n'.join(f"[{a['strategy']}] {a.get('raw_text','')}" for a in attempts)
                new['classification_retry_fullres']=True
                r.clear(); r.update(new)
    return records



def apply_regulatory_classification_gate(records):
    """Bloque l'extraction si la classification n'est pas suffisamment sûre.

    Important : ce garde-fou ne prétend pas garantir 100 % de justesse. Il empêche
    surtout qu'un schéma d'extraction soit appliqué à une page classée AUTRE ou à
    une classification restant sous le seuil après éventuel retry pleine résolution.
    """
    for r in records:
        flags=list(r.get('quality_flags') or [])
        conf=float(r.get('classification_confidence',0) or 0)
        dt=r.get('doc_type')
        review = (dt not in PROMPTS_EXTRACTION) or (conf < CLASSIFICATION_HARD_MIN_CONFIDENCE)
        r['classification_review_required']=bool(review)
        if review:
            flags.append('CLASSIFICATION_REVIEW_REQUIRED')
        r['quality_flags']=list(dict.fromkeys(flags))
    return records


def add_virtual_permit_subdocuments(records):
    """V13.7 : aucune lecture Qwen de la couverture permis.

    Compatibilité : les trois champs PTR restent représentés par un enregistrement
    désactivé avec valeurs nulles. Aucun crop, aucun token, aucun recovery.
    """
    ptr_fields = list(CHAMPS_ATTENDUS.get('PERMIS_TRAVAIL_COUVERTURE') or [])
    physical_cover = [
        r for r in records
        if r.get('doc_type') == 'PERMIS_TRAVAIL_COUVERTURE'
    ]

    # Si une page physique a été classée couverture, on la conserve pour l'audit
    # mais son extraction est explicitement désactivée.
    for r in physical_cover:
        r['raw_data'] = {f: None for f in ptr_fields}
        r['extraction_status'] = 'DISABLED_V13_7'
        r['extraction_error'] = None
        r['extraction_call_count'] = 0
        r['retry_call_count'] = 0
        r['page_recovery_triggered'] = False
        r['recovery_passes'] = 0
        r['extraction_strategies'] = []
        r['extraction_attempts'] = []
        r['quality_flags'] = list(dict.fromkeys(
            list(r.get('quality_flags') or []) + ['PTR_EXTRACTION_DISABLED_V13_7']
        ))

    if physical_cover:
        return records

    # S'il n'existe pas de page physique dédiée, créer uniquement une trace
    # synthétique rattachée à la page TTR. Elle ne déclenche aucun appel Qwen.
    ttr = next((r for r in records if r.get('doc_type') == 'TITRE_TRAVAIL'), None)
    if ttr is None:
        return records

    v = {
        'page_num': ttr.get('page_num'),
        'width': ttr.get('width'),
        'height': ttr.get('height'),
        'white_ratio': ttr.get('white_ratio'),
        'image': ttr.get('image'),
        'doc_type': 'PERMIS_TRAVAIL_COUVERTURE',
        'titre_detecte': 'EXTRACTION_DESACTIVEE_V13_7',
        'bloc_identite_present': False,
        'classification_requalifiee': False,
        'classification_retry_fullres': False,
        'classification_confidence': ttr.get('classification_confidence',0),
        'classification_raw_text': 'AUCUN_APPEL_QWEN_PTR_V13_7',
        'classification_attempts': [],
        'classification_tokens_in': 0,
        'classification_tokens_out': 0,
        'classification_elapsed_s': 0.0,
        'raw_data': {f: None for f in ptr_fields},
        'extraction_status': 'DISABLED_V13_7',
        'extraction_error': None,
        'extraction_raw_text': None,
        'extraction_attempts': [],
        'extraction_strategies': [],
        'extraction_taux_remplissage': 0.0,
        'extraction_tokens_in': 0,
        'extraction_tokens_out': 0,
        'extraction_call_count': 0,
        'retry_call_count': 0,
        'extraction_elapsed_s': 0.0,
        'field_revisions': [],
        'critical_fields_missing': [],
        'field_candidates': {},
        'field_disagreements': [],
        'recovery_history': [],
        'semantic_issues_initial': [],
        'semantic_issues_final': [],
        'coherence_issues_final': [],
        'page_recovery_triggered': False,
        'recovery_passes': 0,
        'field_confidence': {},
        'extraction_confidence_score': None,
        'extraction_confidence_band': 'DISABLED',
        'confidence_method': CONFIDENCE_METHOD,
        'quality_flags': ['PTR_EXTRACTION_DISABLED_V13_7'],
        'is_virtual_subdocument': True,
        'virtual_parent_doc_type': 'TITRE_TRAVAIL',
        'ptr_extraction_disabled': True,
    }
    return records + [v]


def _render_for_strategy(record,pdf_path,strategy_name,crop=None,max_side=None):
    if strategy_name=='STANDARD':
        return record['image']
    return render_page_region(pdf_path,record['page_num']-1,zoom=PDF_ZOOM_HAUTE_DEF,
                              max_side=max_side or IMAGE_MAX_SIZE_HAUTE_DEF,crop=crop)



def _initial_job(record,pdf_path):
    dt=record['doc_type']
    prompt=compact_prompt_for_doc(dt,PROMPTS_EXTRACTION[dt])
    token_cap=int(MAX_NEW_TOKENS_BY_DOC.get(dt,MAX_NEW_TOKENS_EXTRACTION))
    if dt=='TITRE_TRAVAIL':
        # Initial validé historiquement : bloc titre HD. Les recoveries, eux, relisent
        # la page entière afin de gérer un éventuel décalage global.
        crops=crops_planche_permis(record['image']); record['frontiere_planche']=crops['frontiere']
        return {'record':record,'prompt':prompt,
                'image':_render_for_strategy(record,pdf_path,'HD_BLOC_TITRE',crops['titre'],IMAGE_MAX_SIZE_HAUTE_DEF),
                'strategy':'HD_BLOC_TITRE','profile':'HD','mode':'INITIAL',
                'trigger_fields':[],'confirm_fields':[],
                'max_new_tokens':token_cap}
    if dt=='PERMIS_TRAVAIL_COUVERTURE':
        if record.get('is_virtual_subdocument'):
            crops=crops_planche_permis(record['image']); record['frontiere_planche']=crops['frontiere']
            crop=crops['couverture']; name='HD_BLOC_COUVERTURE'
        else:
            crop=(0,1,0,1); name='PAGE_ENTIERE_HD'
        return {'record':record,'prompt':prompt,
                'image':_render_for_strategy(record,pdf_path,name,crop,IMAGE_MAX_SIZE_HAUTE_DEF),
                'strategy':name,'profile':'HD','mode':'INITIAL',
                'trigger_fields':[],'confirm_fields':[],
                'max_new_tokens':token_cap}
    return {'record':record,'prompt':prompt,'image':record['image'],
            'strategy':'STANDARD','profile':'STANDARD','mode':'INITIAL',
            'trigger_fields':[],'confirm_fields':[],
            'max_new_tokens':token_cap}


def _candidate_key(value):
    if value is None: return None
    if isinstance(value,bool): return str(value).lower()
    return re.sub(r'\s+',' ',str(value).strip()).casefold()


def _add_field_candidate(record,field,value,strategy):
    if is_missing_raw(value):
        return
    issue=semantic_issue_for_field(field,value)
    record.setdefault('field_candidates',{}).setdefault(field,[]).append({
        'value':value,
        'strategy':strategy,
        'semantic_valid':issue is None,
        'semantic_issue':issue,
    })


def _merge_job_result(job,output):
    record=job['record']; dt=record['doc_type']; mode=job.get('mode','INITIAL')
    parsed=parse_json_response(output.get('text',''))
    parsed=expand_compact_extraction(parsed,dt)
    expected=CHAMPS_ATTENDUS.get(dt) or []
    trigger=set(job.get('trigger_fields') or [])
    confirm=set(job.get('confirm_fields') or [])
    added=corrected=agreed=disagreed=0
    changed_fields=[]

    attempt_quality=assess_extraction_data(dt,parsed)
    attempt={
        'strategy':job['strategy'],
        'mode':mode,
        'fields_requested':'ALL_PAGE_FIELDS',
        'fields_returned':sorted([k for k,v in parsed.items() if k in expected and not is_missing_raw(v)]),
        'tokens_in':int(output.get('tokens_in',0) or 0),
        'tokens_out':int(output.get('tokens_out',0) or 0),
        'elapsed_s':float(output.get('elapsed_s',0) or 0),
        'is_retry': mode!='INITIAL',
        'semantic_issue_count':len(attempt_quality.get('semantic_issues') or []),
        'coherence_issue_count':len(attempt_quality.get('coherence_issues') or []),
        'critical_missing_count':len(attempt_quality.get('critical_missing') or []),
        'fill_rate':attempt_quality.get('fill_rate'),
    }
    if STORE_QWEN_RAW_TEXT_IN_ATTEMPTS:
        attempt['raw_text']=output.get('text')
    if STORE_PARSED_DATA_IN_ATTEMPTS:
        attempt['parsed_data']=parsed

    record['extraction_attempts'].append(attempt)
    record['extraction_call_count']=int(record.get('extraction_call_count',0) or 0)+1
    if mode!='INITIAL':
        record['retry_call_count']=int(record.get('retry_call_count',0) or 0)+1
        record['recovery_passes']=int(record.get('recovery_passes',0) or 0)+1

    for field in expected:
        if field not in parsed:
            continue
        value=parsed.get(field)
        if is_missing_raw(value):
            continue
        _add_field_candidate(record,field,value,job['strategy'])
        old=record['raw_data'].get(field)
        old_issue=semantic_issue_for_field(field,old) if not is_missing_raw(old) else 'MISSING'
        new_issue=semantic_issue_for_field(field,value)

        if mode=='INITIAL':
            if is_missing_raw(old):
                record['raw_data'][field]=value; added+=1
            continue

        # V13.2 : les 5 champs d'un bloc structurel initial validé sont verrouillés.
        # Toute valeur différente issue d'un recovery reste une preuve d'audit.
        if _v132_locked(record,field):
            if _candidate_key(old)==_candidate_key(value):
                agreed+=1
            else:
                disagreed+=1
                record.setdefault('field_disagreements',[]).append({
                    'field':field,'kept':old,'candidate':value,
                    'strategy':job['strategy'],
                    'reason':'V13_2_STRUCTURAL_LOCK_AUDIT_ONLY'
                })
            continue

        # Recovery : la page entière est relue mais on ne remplace pas aveuglément
        # tous les champs déjà corrects.
        if is_missing_raw(old):
            if new_issue is None:
                record['raw_data'][field]=value; added+=1; changed_fields.append(field)
            continue

        if _candidate_key(old)==_candidate_key(value):
            agreed+=1
            continue

        # Champ encore objectivement problématique après la lecture précédente :
        # une nouvelle valeur sémantiquement plausible peut le corriger.
        if field in trigger and new_issue is None:
            record['raw_data'][field]=value; corrected+=1; changed_fields.append(field)
            record['field_revisions'].append({
                'field':field,'old':old,'new':value,'strategy':job['strategy'],
                'reason':'PAGE_RECOVERY_TRIGGER_FIELD'
            })
            continue

        # Confirmation d'une correction : si la nouvelle lecture diffère alors que
        # les deux valeurs sont plausibles, on n'écrase pas ; on signale le désaccord.
        if field in confirm:
            disagreed+=1
            record.setdefault('field_disagreements',[]).append({
                'field':field,'kept':old,'candidate':value,'strategy':job['strategy'],
                'reason':'RECOVERY_CONFIRMATION_DISAGREEMENT'
            })
            continue

        # Champ non déclencheur : la page a été relue pour le contexte. Une différence
        # n'autorise pas une correction silencieuse.
        if new_issue is None:
            disagreed+=1
            record.setdefault('field_disagreements',[]).append({
                'field':field,'kept':old,'candidate':value,'strategy':job['strategy'],
                'reason':'NON_TRIGGER_FIELD_DISAGREEMENT'
            })

    if mode=='INITIAL' and dt=='TITRE_TRAVAIL':
        _v132_apply_initial_block(record,parsed)
        _v133_fast_safe_lock_from_initial_fields(record)

    attempt['fields_added']=added
    attempt['fields_corrected']=corrected
    attempt['fields_agreed']=agreed
    attempt['fields_disagreed']=disagreed
    attempt['changed_fields']=sorted(set(changed_fields))

    record['extraction_tokens_in']+=int(output.get('tokens_in',0) or 0)
    record['extraction_tokens_out']+=int(output.get('tokens_out',0) or 0)
    record['extraction_elapsed_s']=round(record.get('extraction_elapsed_s',0)+float(output.get('elapsed_s',0) or 0),3)
    record['extraction_taux_remplissage']=taux_remplissage(record['raw_data'],expected)
    record['extraction_strategies'].append({
        'nom':job['strategy'],'mode':mode,'champs_ajoutes':added,
        'champs_corriges':corrected,'accords':agreed,'desaccords':disagreed,
        'taux_apres':record['extraction_taux_remplissage'],
    })
    if mode!='INITIAL':
        record.setdefault('recovery_history',[]).append({
            'strategy':job['strategy'],
            'trigger_fields':sorted(trigger),
            'confirm_fields':sorted(confirm),
            'changed_fields':sorted(set(changed_fields)),
            'issue_report_after':assess_extraction_data(dt,record.get('raw_data') or {}),
        })


def _run_job_chunk(chunk):
    if not chunk: return
    max_new=max(int(j.get('max_new_tokens',MAX_NEW_TOKENS_EXTRACTION)) for j in chunk)
    try:
        _psync(); _bt=_pnow()
        outs=ask_batch_mixed([j['prompt'] for j in chunk],[j['image'] for j in chunk],max_new)
        _psync(); _batch_s=_pnow()-_bt
        for j,o in zip(chunk,outs):
            _merge_job_result(j,o)
            _r=j['record']
            page_pevent(
                _r.get('page_num'),_r.get('doc_type'),
                j.get('mode','EXTRACTION'),j.get('strategy'),
                _batch_s,len(chunk),
                o.get('tokens_in',0),o.get('tokens_out',0),
                note='TEMPS_BATCH_PARTAGE' if len(chunk)>1 else 'TEMPS_PAGE_DIRECT'
            )
    except Exception as exc:
        if len(chunk)>1:
            if is_cuda_oom(exc):
                print(f'⚠️ OOM batch {len(chunk)} -> découpage'); gc.collect(); torch.cuda.empty_cache()
            else:
                print(f'⚠️ Batch {len(chunk)} refusé ({type(exc).__name__}) -> sous-batches')
            mid=len(chunk)//2; _run_job_chunk(chunk[:mid]); _run_job_chunk(chunk[mid:]); return
        raise


def run_extraction_jobs(jobs):
    """V13.7 : conserver le batching agressif qui a donné le meilleur temps.
    Les jobs STANDARD sont groupés ensemble et les jobs HD ensemble.
    """
    std = [j for j in jobs if j.get('profile') == 'STANDARD']
    hd  = [j for j in jobs if j.get('profile') != 'STANDARD']

    for k in range(0, len(std), GPU_BATCH_SIZE_EXTRACTION_STANDARD):
        _run_job_chunk(std[k:k+GPU_BATCH_SIZE_EXTRACTION_STANDARD])

    for k in range(0, len(hd), GPU_BATCH_SIZE_EXTRACTION_HD):
        _run_job_chunk(hd[k:k+GPU_BATCH_SIZE_EXTRACTION_HD])


def _page_recovery_image(record,pdf_path,max_side):
    # Pour un sous-document virtuel de couverture, on relit toute la couverture.
    # Pour tous les vrais documents (TTR, DOM, CTR, CTS), on relit TOUTE LA PAGE.
    if record.get('doc_type')=='PERMIS_TRAVAIL_COUVERTURE' and record.get('is_virtual_subdocument'):
        crop=crops_planche_permis(record['image'])['couverture']
    else:
        crop=(0,1,0,1)
    return _render_for_strategy(record,pdf_path,'PAGE_RECOVERY',crop,max_side)


def _should_recover(report):
    if not ENABLE_PAGE_RECOVERY:
        return False
    if RECOVERY_ON_CRITICAL_MISSING and report.get('critical_missing'):
        return True
    if RECOVERY_ON_SEMANTIC_MISMATCH and report.get('semantic_issues'):
        return True
    if RECOVERY_ON_COHERENCE_ERROR and report.get('coherence_issues'):
        return True
    if RECOVERY_ON_LOW_FILL and report.get('low_fill'):
        return True
    return False


def build_recovery_jobs(records, pdf_path, pass_no=None):
    """V13.7 : UN SEUL recovery possible, directement en page entière 2400 px."""
    jobs = []

    for r in records:
        dt = r.get('doc_type')

        # Couverture PTR : jamais de Qwen en V13.7.
        if dt == 'PERMIS_TRAVAIL_COUVERTURE':
            continue
        if dt not in PROMPTS_EXTRACTION:
            continue

        current = assess_extraction_data(dt, r.get('raw_data') or {})

        # Un bloc TTR structurel déjà verrouillé ne doit pas être dégradé
        # par les contrôles génériques de ses quatre champs.
        if dt == 'TITRE_TRAVAIL' and r.get('ttr_structural_lock'):
            locked = set(TTR_STRUCTURAL_FIELDS_V132)
            current['trigger_fields'] = [
                x for x in (current.get('trigger_fields') or [])
                if x not in locked
            ]
            current['semantic_issues'] = [
                x for x in (current.get('semantic_issues') or [])
                if not (
                    (isinstance(x,dict) and x.get('field') in locked)
                    or x in locked
                )
            ]

        r['semantic_issues_initial'] = list(current.get('semantic_issues') or [])

        structural_need = bool(r.get('structural_confirmation_required'))
        if dt == 'TITRE_TRAVAIL' and structural_need:
            obs_valid = False
            for _o in (r.get('visual_observations') or []):
                _ds = _o.get('ordered_dates_between_duration_and_location') or []
                if (
                    _o.get('strategy') == 'TTR_STRUCTURAL_OBSERVATION'
                    and len(_ds) >= 2
                    and _pair_duration_compatible(
                        _ds[0], _ds[1], _o.get('duration_text')
                    )
                ):
                    obs_valid = True
                    break

            if obs_valid:
                structural_need = False
                current['trigger_fields'] = [
                    x for x in (current.get('trigger_fields') or [])
                    if x not in ('TTR_DATE_DEBUT','TTR_DATE_FIN')
                ]
                current['semantic_issues'] = [
                    x for x in (current.get('semantic_issues') or [])
                    if not (
                        (isinstance(x,dict) and x.get('field')
                         in ('TTR_DATE_DEBUT','TTR_DATE_FIN'))
                        or x in ('TTR_DATE_DEBUT','TTR_DATE_FIN')
                    )
                ]

        # V13.6/V13.7 fast-safe :
        # un simple warning sémantique non critique ne suffit pas à relancer Qwen.
        need = False
        if ENABLE_PAGE_RECOVERY:
            if RECOVERY_ON_CRITICAL_MISSING and current.get('critical_missing'):
                need = True
            if RECOVERY_ON_COHERENCE_ERROR and current.get('coherence_issues'):
                need = True
            if RECOVERY_ON_LOW_FILL and current.get('low_fill'):
                need = True
            if structural_need:
                need = True

        if not need:
            continue

        r['page_recovery_triggered'] = True
        trigger = list(current.get('trigger_fields') or [])
        if structural_need and dt == 'TITRE_TRAVAIL':
            trigger = sorted(set(trigger) | {'TTR_DATE_DEBUT','TTR_DATE_FIN'})

        jobs.append({
            'record': r,
            'prompt': compact_prompt_for_doc(
                dt, build_page_recovery_prompt(dt,current,2400)
            ),
            'image': _page_recovery_image(
                r,pdf_path,IMAGE_MAX_SIZE_RECOVERY_2
            ),
            'strategy': 'PAGE_RECOVERY_2400_SINGLE',
            'profile': 'HD',
            'mode': 'RECOVERY_2400_SINGLE',
            'trigger_fields': trigger,
            'confirm_fields': [],
            'max_new_tokens': int(
                MAX_NEW_TOKENS_RECOVERY_BY_DOC.get(
                    dt,MAX_NEW_TOKENS_RECOVERY
                )
            ),
        })

    return jobs


def extract_classified_pages(records,pdf_path):
    applicable = [
        r for r in records
        if r.get('doc_type') in PROMPTS_EXTRACTION
        and r.get('doc_type') != 'PERMIS_TRAVAIL_COUVERTURE'
        and not (
            BLOCK_EXTRACTION_ON_CLASSIFICATION_CONFLICT
            and r.get('classification_review_required')
        )
    ]

    # 1) Extraction initiale.
    run_extraction_jobs([_initial_job(r,pdf_path) for r in applicable])

    # 2) TTR : observation structurelle uniquement si le bloc initial n'est
    # pas déjà verrouillé par la logique Fast-Safe.
    run_ttr_structural_observations(applicable,pdf_path)

    for r in applicable:
        _v134_lock_from_specialized_observation(r)

    # 3) V13.7 : un seul recovery possible, directement à 2400 px.
    recovery = build_recovery_jobs(applicable,pdf_path)
    run_extraction_jobs(recovery)

    # 4) Résolution TTR finale sur les preuves de LA MEME PAGE.
    for r in records:
        resolve_ttr_structural_dates(r)

    for r in records:
        finalize_extraction_record(r)

    return records


def print_call_diagnostics(records):
    if not PRINT_CALL_DIAGNOSTICS:
        return
    print('\n--- Diagnostic appels Qwen par page V13.7 ---')
    for r in records:
        if r.get('doc_type') not in PROMPTS_EXTRACTION:
            continue
        strategies=[a.get('strategy') for a in (r.get('extraction_attempts') or [])]
        print(
            f"page={r.get('page_num')} type={r.get('doc_type')} "
            f"classif={r.get('classification_confidence')} "
            f"calls={r.get('extraction_call_count',0)} retries={r.get('retry_call_count',0)} "
            f"fill={r.get('extraction_taux_remplissage')} "
            f"confidence={r.get('extraction_confidence_score')}({r.get('extraction_confidence_band')}) "
            f"critical={r.get('critical_fields_missing')} "
            f"semantic={len(r.get('semantic_issues_final') or [])} "
            f"coherence={len(r.get('coherence_issues_final') or [])} "
            f"strategies={strategies}"
        )


def _json_safe_record(record):
    return {k:v for k,v in record.items() if k!='image'}


def process_pdf(pdf_path):
    profiler_reset()
    t0=time.time(); log(f'📁 {pdf_path.name}')
    with pstage("PDF_TO_PAGES",pdf=pdf_path.name):
        pages=pdf_to_pages(pdf_path)
    with pstage("CLASSIFICATION_TOTAL",pages=len(pages)):
        records=apply_regulatory_classification_gate(classify_pages(pages))
    records=add_virtual_permit_subdocuments(records)
    with pstage("EXTRACTION_TOTAL",pages=len(pages)):
        records=extract_classified_pages(records,pdf_path)
    print_call_diagnostics(records)
    tokens_in=sum(r.get('classification_tokens_in',0)+r.get('extraction_tokens_in',0) for r in records)
    tokens_out=sum(r.get('classification_tokens_out',0)+r.get('extraction_tokens_out',0) for r in records)
    elapsed=round(time.time()-t0,3)
    physical_records=[r for r in records if not r.get('is_virtual_subdocument')]
    present_doc_types=sorted(set(
        r.get('doc_type') for r in physical_records
        if r.get('doc_type') and r.get('doc_type')!='AUTRE'
    ))
    expected_core=['ENGAGEMENT_DOMICILIATION','CONTRAT_TRAVAIL','CONTRAT_SPECIFIQUE','TITRE_TRAVAIL']
    page_presence={
        'present_doc_types':present_doc_types,
        'missing_core_doc_types':[x for x in expected_core if x not in present_doc_types],
        'classification_review_pages':[
            r.get('page_num') for r in physical_records if r.get('classification_review_required')
        ],
        'physical_pages':len(pages),
        'note':'Absence page != champ OCR vide. La Partie 2 exploite cette distinction.'
    }

    dossier={
        'schema_version':SCHEMA_VERSION,
        'field_schema_hash':FIELD_SCHEMA_HASH,
        'field_schema':FIELD_SCHEMA,
        'source_file':pdf_path.name,
        'source_sha256':sha256_file(pdf_path),
        'pipeline_version':PIPELINE_VERSION,
        'extraction_engine':{
            'model':'Qwen3.6-27B-FP8','model_path':MODEL_PATH,'flash_attn':False,
            'classification_policy':'MANDATORY_VLM_PER_PAGE_NO_PAGE_ORDER',
            'classification_hard_min_confidence':CLASSIFICATION_HARD_MIN_CONFIDENCE,
            'standard_max_side':IMAGE_MAX_SIZE,'classification_max_side':IMAGE_MAX_SIZE_CLASSIFICATION,
            'hd_max_side':IMAGE_MAX_SIZE_HAUTE_DEF,
            'recovery_1_max_side':None,
            'recovery_2_max_side':IMAGE_MAX_SIZE_RECOVERY_2,
            'recovery_policy':'V13_7_SINGLE_WHOLE_PAGE_2400_ONLY',
            'ptr_extraction_policy':'DISABLED_NO_QWEN_KEEP_3_FIELDS_NULL',
            'confidence_method':CONFIDENCE_METHOD,
            'confidence_note':'Operational indicator; not native Qwen log-probability',
            'created_at':datetime.now().isoformat(timespec='seconds'),
        },
        'stats':{
            'pages':len(pages),'page_records':len(records),
            'virtual_subdocuments':sum(bool(r.get('is_virtual_subdocument')) for r in records),
            'classification_review_required':sum(bool(r.get('classification_review_required')) for r in records),
            'classification_calls':sum(len(r.get('classification_attempts') or []) for r in records),
            'extraction_calls':sum(int(r.get('extraction_call_count',0) or 0) for r in records),
            'retry_calls':sum(int(r.get('retry_call_count',0) or 0) for r in records),
            'pages_recovered':sum(bool(r.get('page_recovery_triggered')) for r in records),
            'structural_observation_calls':sum(
                int(r.get('structural_observation_call_count',0) or 0) for r in records
            ),
            'qwen_calls_total':sum(
                len(r.get('classification_attempts') or [])
                + int(r.get('extraction_call_count',0) or 0)
                + int(r.get('structural_observation_call_count',0) or 0)
                for r in records
            ),
            'fast_safe_ttr_locked_initial':sum(
                r.get('ttr_fast_safe_status') in (
                    'LOCKED_FROM_INITIAL_FIELDS',
                    'LOCKED_FROM_INITIAL_STRUCTURAL_OBJECT'
                ) for r in records
            ),
            'extraction_passes':sum(len(r.get('extraction_strategies') or []) for r in records),
            'field_revisions':sum(len(r.get('field_revisions') or []) for r in records),
            'field_disagreements':sum(len(r.get('field_disagreements') or []) for r in records),
            'pages_confidence_high':sum(r.get('extraction_confidence_band')=='HIGH' for r in records),
            'pages_confidence_medium':sum(r.get('extraction_confidence_band')=='MEDIUM' for r in records),
            'pages_confidence_low':sum(r.get('extraction_confidence_band')=='LOW' for r in records),
            'tokens_in':int(tokens_in),'tokens_out':int(tokens_out),'tokens_total':int(tokens_in+tokens_out),
            'elapsed_s':elapsed,
        },
        'page_presence':page_presence,
        'page_records':[_json_safe_record(r) for r in records],
    }
    canonical_checkpoint_path(pdf_path).write_text(json.dumps(dossier,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    profiler_finish(OUTPUT_ROOT)
    return dossier

print('✅ Partie 1 V13.7.1 Fast-Safe prête — early-stop déterministe, fallback V13.2 conservé')


In [ ]:
# Régressions V13 — TTR
assert semantic_issue_for_field('TTR_DUREE', 'TAGE / LAMINOR 2') == 'EXPECTED_DURATION'
assert semantic_issue_for_field('TTR_DUREE', '2 ANS, 0 JOURS') is None
assert semantic_issue_for_field('TTR_DUREE', '24 MOIS') is None

_test_ttr = {
    'doc_type':'TITRE_TRAVAIL',
    'raw_data':{'TTR_DATE_DEBUT':None, 'TTR_DATE_FIN':None},
    'field_candidates':{
        'TTR_DATE_DEBUT':[{'value':'14/04/2025','strategy':'HD_BLOC_TITRE'}],
        'TTR_DATE_FIN':[{'value':'13/04/2027','strategy':'HD_BLOC_TITRE'}],
    },
    'quality_flags':[],
    'field_revisions':[],
}
resolve_ttr_structural_dates(_test_ttr)
assert _test_ttr['raw_data']['TTR_DATE_DEBUT'] == '14/04/2025'
assert _test_ttr['raw_data']['TTR_DATE_FIN'] == '13/04/2027'
print('✅ V13 P1 : fausse durée détectée sans effacer les bonnes dates TTR')


In [ ]:
# Régression FINAL V12 — aucune dépendance aux durées en Partie 1
_r={
    'doc_type':'TITRE_TRAVAIL',
    'raw_data':{
        'TTR_DUREE':'TAGE / LAMINOR 2',
        'TTR_DATE_DEBUT':None,
        'TTR_DATE_FIN':None
    },
    'field_candidates':{
        'TTR_DATE_DEBUT':[{'value':'14/04/2025','strategy':'HD_BLOC_TITRE'}],
        'TTR_DATE_FIN':[{'value':'13/04/2027','strategy':'HD_BLOC_TITRE'}],
    },
    'quality_flags':[]
}
resolve_ttr_structural_dates(_r)
assert _r['raw_data']['TTR_DATE_DEBUT']=='14/04/2025', _r
assert _r['raw_data']['TTR_DATE_FIN']=='13/04/2027', _r
# La Partie 1 ne corrige ni ne supprime TTR_DUREE.
assert _r['raw_data']['TTR_DUREE']=='TAGE / LAMINOR 2', _r
assert _r['ttr_structural_date_resolution']=='RESOLVED_BY_DATE_ORDER_ONLY', _r
print('✅ FINAL V12 : dates TTR indépendantes de la durée; contrôle durée réservé à Partie 2')


In [ ]:
# Régression FINAL V12 — cas réel observé
_r={
    'doc_type':'TITRE_TRAVAIL',
    'raw_data':{'TTR_DUREE':'TAGE / LAMINOR 2','TTR_DATE_DEBUT':None,'TTR_DATE_FIN':None},
    'field_candidates':{
        'TTR_DUREE':[{'value':'TAGE / LAMINOR 2','strategy':'HD_BLOC_TITRE'}],
        'TTR_DATE_DEBUT':[{'value':'14/04/2025','strategy':'HD_BLOC_TITRE'}],
        'TTR_DATE_FIN':[{'value':'13/04/2027','strategy':'HD_BLOC_TITRE'}],
    },
    'quality_flags':[]
}
resolve_ttr_structural_dates(_r)
assert _r['raw_data']['TTR_DATE_DEBUT']=='14/04/2025', _r
assert _r['raw_data']['TTR_DATE_FIN']=='13/04/2027', _r
assert _r['raw_data']['TTR_DUREE'] is None, _r
assert _r['ttr_structural_date_resolution']=='RESOLVED_DATES_ORDER_OK_DURATION_UNREADABLE', _r
print('✅ FINAL V12 : dates conservées; durée mal alignée isolée pour revue')


## 11. Tests techniques V9.3 sans inférence GPU


In [ ]:

assert sum(len(v) for v in CHAMPS_ATTENDUS.values()) == 99
assert 'TTR_NUMERO_PERMIS' in CHAMPS_ATTENDUS['TITRE_TRAVAIL']
assert IMAGE_MAX_SIZE_RECOVERY_1 >= 2000
assert IMAGE_MAX_SIZE_RECOVERY_2 >= 2400
assert value_is_suspect('DOM_SALAIRE_NET_MENSUEL','23.340.43') is False

# 1) Exemple utilisateur : un lieu dans un champ date doit déclencher recovery.
assert semantic_issue_for_field('TTR_DATE_DEBUT','ORAN') == 'EXPECTED_DATE'
# 2) Une date dans un lieu doit aussi être détectée.
assert semantic_issue_for_field('TTR_LIEU_TRAVAIL','14/08/2025') == 'EXPECTED_TEXT_GOT_DATE'
# 3) Une date valide reste acceptée sans normalisation.
assert semantic_issue_for_field('TTR_DATE_DEBUT','14/08/2025') is None
# 4) Ordre chronologique impossible -> cohérence page.
_test={'TTR_DATE_DEBUT':'13/08/2027','TTR_DATE_FIN':'14/08/2025','TTR_DUREE':'2 ANS'}
_issues=coherence_issues_for_data('TITRE_TRAVAIL',_test)
assert any(x['code']=='WORK_PERMIT_DATE_ORDER' for x in _issues)
# 5) Cas cohérent 2 ans ~ 14/08/2025 -> 13/08/2027.
_test2={'TTR_DATE_DEBUT':'14/08/2025','TTR_DATE_FIN':'13/08/2027','TTR_DUREE':'2 ANS, 0 JOURS'}
assert not coherence_issues_for_data('TITRE_TRAVAIL',_test2)
# 6) Un titre complet/plausible ne doit pas déclencher par le simple fait du layout.
_test_title={k:None for k in CHAMPS_ATTENDUS['TITRE_TRAVAIL']}
_test_title.update({
    'TTR_NUMERO_PERMIS':'25-00005959 / 31-25-001316',
    'TTR_POSTE':"CONTREMAITRE D'ENTRETIEN ELECTRONIQUE",
    'TTR_DUREE':'2 ANS, 0 JOURS','TTR_DATE_DEBUT':'14/08/2025','TTR_DATE_FIN':'13/08/2027',
    'TTR_LIEU_TRAVAIL':'POLE ECONOMIQUE GOURIRATE BETHIOUA','TTR_EMPLOYEUR':'CLIENT IRONSTEEL IND SPA',
    'TTR_NOM':'ABDUL','TTR_PRENOM':'SHUKUR','TTR_DATE_NAISSANCE':'05/06/1979',
    'TTR_NATIONALITE':'INDIENNE','TTR_DATE_ENTREE_ALGERIE':'05/08/2025',
})
_rep=assess_extraction_data('TITRE_TRAVAIL',_test_title)
assert not _rep['semantic_issues']
assert not _rep['coherence_issues']

print('✅ Tests FINAL V12 : 99 champs, semantic mismatch, dates, layout recovery OK')


# 7) Régression observée : 02/01/2023 -> 03/01/2026 n'est PAS cohérent avec 2 ans.
_ok,_reason=_ttr_triplet_is_coherent('02/01/2023','03/01/2026','2 ANS, 0 JOURS')
assert _ok is False, _reason

# 8) Cas attendu sur la page test : 03/01/2026 -> 02/01/2028 est cohérent avec 2 ans.
_ok,_reason=_ttr_triplet_is_coherent('03/01/2026','02/01/2028','2 ANS, 0 JOURS')
assert _ok is True, _reason

# 9) Une paire inversée reste rejetée.
_ok,_reason=_ttr_triplet_is_coherent('02/01/2028','03/01/2026','2 ANS, 0 JOURS')
assert _ok is False, _reason

print('✅ Régression FINAL V12 : triplet TTR durée/début/fin contrôlé')


## 12. Exécution — validation V9.3 sur 10 dossiers avant batch complet


In [ ]:
if not pdfs:
    print('⚠️ Aucun PDF dans', INPUT_DIR)
else:
    results=[]; errors=[]
    for i,pdf in enumerate(pdfs,1):
        print(f'\n[{i}/{len(pdfs)}] {pdf.name}')
        existing=load_existing_checkpoint(pdf)
        if existing is not None:
            print('↪ checkpoint V9.3 RAW valide réutilisé')
            d=existing; statut='REPRIS'
        else:
            try:
                d=process_pdf(pdf); statut='TRAITE'
            except Exception as exc:
                log(f'❌ {pdf.name} : {repr(exc)}')
                errors.append({'source_file':pdf.name,'error':repr(exc),'date':datetime.now().isoformat(timespec='seconds')})
                continue
        s=d.get('stats') or {}
        results.append({'source_file':d.get('source_file'),'status':statut,'sha256':d.get('source_sha256'),
                        'pages':s.get('pages'),'page_records':s.get('page_records'),
                        'tokens_total':s.get('tokens_total'),'elapsed_s':s.get('elapsed_s'),
                        'qwen_calls_total':s.get('qwen_calls_total'),'extraction_calls':s.get('extraction_calls'),'retry_calls':s.get('retry_calls'),'pages_recovered':s.get('pages_recovered'),'pages_confidence_high':s.get('pages_confidence_high'),'pages_confidence_medium':s.get('pages_confidence_medium'),'pages_confidence_low':s.get('pages_confidence_low'),
                        'json_path':str(canonical_checkpoint_path(pdf))})
        print(f"✅ {statut} | pages={s.get('pages')} | Qwen calls={s.get('qwen_calls_total')} | extraction={s.get('extraction_calls')} | retries={s.get('retry_calls')} | recovered_pages={s.get('pages_recovered')} | tokens={s.get('tokens_total')} | temps={s.get('elapsed_s')}s")

    manifest={'schema_version':SCHEMA_VERSION,'pipeline_version':PIPELINE_VERSION,
              'field_schema_hash':FIELD_SCHEMA_HASH,'generated_at':datetime.now().isoformat(timespec='seconds'),
              'results':results,'errors':errors}
    MANIFEST_PATH.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    pd.DataFrame(results).to_csv(INDEX_CSV_PATH,index=False,encoding='utf-8-sig')
    print('\n✅ Manifest :',MANIFEST_PATH)
    print('✅ Index    :',INDEX_CSV_PATH)
    print('✅ JSON RAW :',JSON_DIR)


## Contrat de sortie V13.2

Le schéma `DOM_EXTRACTION_V1`, le hash et les **99 champs RAW restent inchangés** afin de rester compatibles avec la Partie 2.

V9.3 ajoute uniquement des métadonnées d'audit autour du RAW :
- `semantic_issues_initial` / `semantic_issues_final` ;
- `coherence_issues_final` ;
- `page_recovery_triggered`, `recovery_history` ;
- `field_candidates`, `field_disagreements`, `field_revisions` ;
- `field_confidence` ;
- `extraction_confidence_score` / `extraction_confidence_band` ;
- `confidence_method = OPERATIONAL_CONSENSUS_SEMANTIC_V1`.

La confiance est **opérationnelle et explicable**, pas une probabilité native Qwen. Une page problématique est relue seule, à 2000 puis éventuellement 2400 px ; les autres pages du PDF ne sont pas retraitées.


In [ ]:

# V13 — tests de régression evidence-first (données fictives)
_r={
 'doc_type':'TITRE_TRAVAIL',
 'raw_data':{'TTR_DATE_DEBUT':'01/01/2025','TTR_DATE_FIN':'31/12/2026'},
 'field_candidates':{
   'TTR_DATE_DEBUT':[
      {'value':'01/01/2025','strategy':'HD_BLOC_TITRE'},
      {'value':'01/01/2025','strategy':'TTR_STRUCTURAL_OBSERVATION'}],
   'TTR_DATE_FIN':[
      {'value':'31/12/2026','strategy':'HD_BLOC_TITRE'},
      {'value':'31/12/2026','strategy':'TTR_STRUCTURAL_OBSERVATION'}],
 },
 'quality_flags':[]
}
resolve_ttr_structural_dates(_r)
assert _r['raw_data']['TTR_DATE_DEBUT']=='01/01/2025'
assert _r['raw_data']['TTR_DATE_FIN']=='31/12/2026'
assert _r['ttr_structural_date_resolution']=='RESOLVED_BY_SAME_PAGE_CONSENSUS'
print("✅ V13 P1 : observation structurelle + consensus même page")


In [ ]:

# V13.2 — régressions rotation + preuve structurelle (valeurs fictives)
assert _pair_duration_compatible('01/01/2025','31/12/2026','2 ANS, 0 JOURS')
assert not _pair_duration_compatible('01/01/2025','02/01/2025','2 ANS, 0 JOURS')

_test={
 'doc_type':'TITRE_TRAVAIL',
 'raw_data':{'TTR_DATE_DEBUT':'02/01/2025','TTR_DATE_FIN':'03/01/2025'},
 'visual_observations':[{
   'strategy':'TTR_STRUCTURAL_OBSERVATION',
   'duration_text':'2 ANS, 0 JOURS',
   'ordered_dates_between_duration_and_location':['01/01/2025','31/12/2026']
 }],
 'field_candidates':{
   'TTR_DATE_DEBUT':[{'value':'02/01/2025','strategy':'PAGE_RECOVERY_2000_LAYOUT'}],
   'TTR_DATE_FIN':[{'value':'03/01/2025','strategy':'PAGE_RECOVERY_2000_LAYOUT'}],
 },
 'quality_flags':[]
}
resolve_ttr_structural_dates(_test)
assert _test['raw_data']['TTR_DATE_DEBUT']=='01/01/2025'
assert _test['raw_data']['TTR_DATE_FIN']=='31/12/2026'
assert _test['ttr_structural_date_resolution']=='RESOLVED_BY_STRUCTURAL_EVIDENCE'
assert _test['ttr_rejected_date_candidates'][0]['reason']=='INCOMPATIBLE_WITH_DURATION_AND_VISUAL_STRUCTURE'
print("✅ V13.2 : preuve structurelle souveraine + deskew déterministe intégrés")


## V13.2 — comportement ajouté

- Le premier appel Qwen TTR reconstruit le bloc `POSTE → DUREE → DATE_DEBUT → DATE_FIN → LIEU_TRAVAIL`.
- `POSTE` et `LIEU_TRAVAIL` peuvent avoir plusieurs lignes.
- Le décalage vertical global des valeurs est explicitement pris en compte.
- Si le bloc initial est cohérent (ordre des dates + durée locale), les 5 champs sont verrouillés.
- Les recoveries 2000/2400 contradictoires restent en audit et ne remplacent pas ces valeurs.
- L'appel `TTR_STRUCTURAL_OBSERVATION` supplémentaire devient un fallback uniquement.
- Rotation/deskew de V13.1 conservé.
- Aucune autre page ne corrige le RAW de cette page en Partie 1.
- Les 99 champs métier et le hash de schéma restent inchangés.


In [ ]:

# V13.2 — tests ciblés
_a={"TTR_STRUCTURAL_BLOCK":{
 "poste_lines":["CONTREMAITRE MECANICIEN"],"duree":"2 ANS, 0 JOURS",
 "ordered_dates":["16/11/2025","15/11/2027"],
 "lieu_lines":["POLE ECONOMIQUE PLATEAU GOURIRATE B","ETHIOUA"],
 "global_vertical_shift_suspected":False,"notes":[]}}
_v=_v132_read_block(_a); _ok,_why=_v132_validate_block(_v)
assert _ok,_why
assert _v["TTR_DATE_DEBUT"]=="16/11/2025"
assert _v["TTR_DATE_FIN"]=="15/11/2027"

_b={"TTR_STRUCTURAL_BLOCK":{
 "poste_lines":["CONTREMAITRE DES ROULEAUX","/ EMPAQUE"],"duree":"2 ANS, 0 JOURS",
 "ordered_dates":["14/04/2025","13/04/2027"],
 "lieu_lines":["POLE ECONOMIQUE GOURIRATE","BETHIOUA"],
 "global_vertical_shift_suspected":True,"notes":[]}}
_v=_v132_read_block(_b); _ok,_why=_v132_validate_block(_v)
assert _ok,_why
assert "EMPAQUE" in _v["TTR_POSTE"]
assert _v["TTR_DATE_DEBUT"]=="14/04/2025"

_bad={"TTR_STRUCTURAL_BLOCK":{
 "poste_lines":["CONTREMAITRE"],"duree":"TAGE / LAMINOR 2",
 "ordered_dates":["2 ANS, 0 JOURS","14/04/2025"],
 "lieu_lines":["13/04/2027 POLE ECONOMIQUE"],
 "global_vertical_shift_suspected":True,"notes":[]}}
_v=_v132_read_block(_bad); _ok,_why=_v132_validate_block(_v)
assert not _ok
assert "DURATION_NOT_RECOGNIZED" in _why
print("✅ V13.2 structural-first tests OK")


## V13.3 Fast-Safe

Objectif : réduire le temps sans réduire la qualité.

- Même modèle Qwen et mêmes résolutions que V13.2.
- Après le premier appel TTR, Python valide directement les 5 valeurs
  `POSTE → DUREE → DATE_DEBUT → DATE_FIN → LIEU_TRAVAIL`.
- Si ce bloc est strictement cohérent, il est verrouillé : pas d'appel
  `TTR_STRUCTURAL_OBSERVATION`, ni de recovery 2000/2400 pour ces champs.
- Si le bloc n'est pas strictement cohérent, aucun raccourci : le fallback
  complet V13.2 reste actif.
- Le recovery 2400 des autres pages est conservé si une anomalie subsiste ou
  si le recovery 2000 a corrigé un champ critique.
- Les modifications non critiques ne déclenchent plus à elles seules le 2400.
- Les 99 champs métier et le hash du schéma restent inchangés.


In [ ]:

# V13.3 — tests Fast-Safe
_good={
 "doc_type":"TITRE_TRAVAIL",
 "raw_data":{
  "TTR_POSTE":"CONTREMAITRE MECANICIEN",
  "TTR_DUREE":"2 ANS, 0 JOURS",
  "TTR_DATE_DEBUT":"16/11/2025",
  "TTR_DATE_FIN":"15/11/2027",
  "TTR_LIEU_TRAVAIL":"POLE ECONOMIQUE PLATEAU GOURIRATE B ETHIOUA"
 }}
_ok,_why,_=_v133_validate_initial_raw_ttr(_good)
assert _ok,_why

_multiline={
 "doc_type":"TITRE_TRAVAIL",
 "raw_data":{
  "TTR_POSTE":"CONTREMAITRE DES ROULEAUX / EMPAQUE",
  "TTR_DUREE":"2 ANS, 0 JOURS",
  "TTR_DATE_DEBUT":"14/04/2025",
  "TTR_DATE_FIN":"13/04/2027",
  "TTR_LIEU_TRAVAIL":"POLE ECONOMIQUE GOURIRATE BETHIOUA"
 }}
_ok,_why,_=_v133_validate_initial_raw_ttr(_multiline)
assert _ok,_why

_shifted={
 "doc_type":"TITRE_TRAVAIL",
 "raw_data":{
  "TTR_POSTE":"CONTREMAITRE DES ROULEAUX",
  "TTR_DUREE":"TAGE / LAMINOR 2",
  "TTR_DATE_DEBUT":"2 ANS, 0 JOURS",
  "TTR_DATE_FIN":"14/04/2025",
  "TTR_LIEU_TRAVAIL":"13/04/2027 POLE ECONOMIQUE"
 }}
_ok,_why,_=_v133_validate_initial_raw_ttr(_shifted)
assert not _ok
assert "DURATION_NOT_RECOGNIZED" in _why
assert "LOCATION_STARTS_WITH_DATE" in _why

print("✅ V13.3 Fast-Safe regression tests OK")


## V13.4 — Fast-Safe 2

Deux optimisations supplémentaires sans réduction du modèle ni de la résolution :

1. Quand le premier TTR reste douteux, `TTR_STRUCTURAL_OBSERVATION` devient une
   véritable source spécialisée : si poste/durée/dates/lieu forment un bloc
   cohérent, le bloc est verrouillé **avant** le recovery générique 2000.
2. Le recovery 2400 est réservé aux anomalies critiques ou de cohérence.
   Une anomalie sémantique non critique seule est conservée pour audit/validation
   et ne provoque plus automatiquement un troisième appel Qwen.

Le fallback complet reste actif si la lecture spécialisée n'est pas suffisamment sûre.


## V13.4 PROFILER

Utiliser idéalement **un seul PDF**.

Le notebook affiche en temps réel les étapes et mesure :
`PDF_TO_PAGES`, `CLASSIFICATION_TOTAL`, `EXTRACTION_TOTAL`, `CHAT_TEMPLATE`,
`PROCESSOR`, `TRANSFER_GPU`, `MODEL_GENERATE` et `DECODE`.

À la fin, regarder surtout `MODEL_GENERATE` dans le résumé. Deux fichiers sont
créés : `profiler_events_v13_7_1.csv` et `profiler_summary_v13_7_1.csv`.


In [ ]:

# =====================================================================
# V13.5 — REGRESSION COMPACT OUTPUT
# =====================================================================
for _dt,_fields in CHAMPS_ATTENDUS.items():
    _aliases=COMPACT_ALIAS_BY_DOC[_dt]
    assert len(_aliases)==len(_fields)
    assert len(set(_aliases.values()))==len(_fields)
    _fake={_aliases[f]:f"VALUE_{i}" for i,f in enumerate(_fields)}
    _expanded=expand_compact_extraction(_fake,_dt)
    assert all(_expanded[f]==f"VALUE_{i}" for i,f in enumerate(_fields))

# L'objet structurel TTR doit survivre au remapping.
_dt='TITRE_TRAVAIL'
_a=COMPACT_ALIAS_BY_DOC[_dt]
_fake={
    _a['TTR_DATE_DEBUT']:'01/01/2025',
    _a['TTR_DATE_FIN']:'31/12/2026',
    'TTR_STRUCTURAL_BLOCK':{
        'poste_lines':['POSTE TEST'],
        'duree':'2 ANS, 0 JOURS',
        'ordered_dates':['01/01/2025','31/12/2026'],
        'lieu_lines':['LIEU TEST'],
        'global_vertical_shift_suspected':False,
        'notes':[]
    }
}
_e=expand_compact_extraction(_fake,_dt)
assert _e['TTR_DATE_DEBUT']=='01/01/2025'
assert _e['TTR_DATE_FIN']=='31/12/2026'
assert 'TTR_STRUCTURAL_BLOCK' in _e

assert sum(len(v) for v in CHAMPS_ATTENDUS.values())==99
assert FIELD_SCHEMA_HASH=='da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e'
print("✅ V13.5 compact-output : mapping 99 champs + TTR structural OK")


## V13.5 — optimisation de la génération

Cette version cible le goulot mesuré par V13.4 PROFILER : `model.generate()`.

Modifications :
- clés Qwen courtes `f01`, `f02`, etc. pendant l'inférence, remappées
  immédiatement vers les **99 noms de champs canoniques** dans `raw_data` ;
- aucun changement du schéma `DOM_EXTRACTION_V1` ni du hash ;
- budgets de génération adaptés au type de document ;
- batching équilibré par longueur : DOM séparé de CTR/CTS, et surtout
  `PERMIS_TRAVAIL_COUVERTURE` séparé du TTR ;
- observation structurelle TTR raccourcie aux seules informations utilisées
  pour le verrou structurel ;
- profiler conservé et erreur `ROOT` corrigée.

La logique de validation, les recoveries et les contrôles V13.4 restent actifs.


In [ ]:

# =====================================================================
# V13.7 — REGRESSION / CONTRAT
# =====================================================================
assert FIELD_SCHEMA_HASH == 'da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e'
assert sum(len(v) for v in CHAMPS_ATTENDUS.values()) == 99
assert ENABLE_VIRTUAL_PERMIT_COVER is False
assert IMAGE_MAX_SIZE_RECOVERY_2 == 2400

_ptr = CHAMPS_ATTENDUS['PERMIS_TRAVAIL_COUVERTURE']
assert len(_ptr) == 3

# Un seul type de stratégie recovery doit être actif dans le moteur V13.7.
_src_engine = ''.join(In[0:0]) if False else 'runtime_check_skipped'
print("✅ V13.7 contrat : 99 champs inchangés")
print("✅ PTR : 3 champs conservés, extraction Qwen désactivée")
print("✅ Recovery : un seul passage 2400")
print("✅ Profiler : global + détail par page")


## V13.7 — changements ciblés

Cette version conserve le modèle, la résolution initiale, le deskew, le schéma
`DOM_EXTRACTION_V1`, le hash et les 99 champs.

Changements uniquement :
1. `PERMIS_TRAVAIL_COUVERTURE` n'est plus envoyé à Qwen. Les trois `PTR_*`
   restent présents avec `null` et la trace `PTR_EXTRACTION_DISABLED_V13_7`.
2. Suppression du recovery 2000 et de la confirmation supplémentaire.
   Une page problématique peut avoir **un seul recovery**, directement en
   page entière **2400 px**.
3. Batching agressif conservé.
4. Profiler lisible par page et par étape. Pour un batch multi-pages, le vrai
   temps GPU est affiché comme `shared_batch_s`; `allocated_s` est seulement
   une imputation additive du temps partagé, pas une mesure GPU isolée.
